# 03.5.1 FastF1 Data Reproduction

**Purpose**: Reproduce Kaggle-equivalent data from FastF1 sources, starting with 2024 validation, then producing 2025 data.

**Scope**: This notebook maps all required columns from FastF1 data to match the structure of `master_races_clean.csv`, calculates standings, and handles sprint results in the same row structure.

**Output**: 
- Validated 2024 reproduction (compared against master)
- 2025 production data (to append to master for rolling calculations)


## Setup: Imports, Paths, and Helper Functions


In [60]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

# Get project root (works whether running from notebooks/ or F1/ folder)
PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

# Paths
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_DIR / 'raw'
PROCESSED_DATA_DIR = DATA_DIR / 'processed'
KAGGLE_DIR = RAW_DATA_DIR / 'kaggle'
FASTF1_DIR = RAW_DATA_DIR / 'fastf1_2018plus'

# Create output directories if needed
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project Structure:")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  KAGGLE_DIR: {KAGGLE_DIR}")
print(f"  FASTF1_DIR: {FASTF1_DIR}")
print(f"  PROCESSED_DATA_DIR: {PROCESSED_DATA_DIR}")
print()

# Verify directories exist
assert KAGGLE_DIR.exists(), f"Kaggle data directory not found: {KAGGLE_DIR}"
assert FASTF1_DIR.exists(), f"FastF1 data directory not found: {FASTF1_DIR}"
print("✓ All directories exist")


Project Structure:
  PROJECT_ROOT: C:\Users\Erik Viljamaa\Downloads\projects\f1-podium-predictor
  KAGGLE_DIR: C:\Users\Erik Viljamaa\Downloads\projects\f1-podium-predictor\data\raw\kaggle
  FASTF1_DIR: C:\Users\Erik Viljamaa\Downloads\projects\f1-podium-predictor\data\raw\fastf1_2018plus
  PROCESSED_DATA_DIR: C:\Users\Erik Viljamaa\Downloads\projects\f1-podium-predictor\data\processed

✓ All directories exist


In [61]:
# Load lookup tables
print("Loading lookup tables...")

# Circuits lookup
circuits_df = pd.read_csv(KAGGLE_DIR / 'circuits.csv', low_memory=False)
print(f"  ✓ Loaded circuits.csv: {len(circuits_df)} circuits")

# Drivers lookup
drivers_df = pd.read_csv(KAGGLE_DIR / 'drivers.csv', low_memory=False)
print(f"  ✓ Loaded drivers.csv: {len(drivers_df)} drivers")

# Status lookup (for mapping FastF1 Status text to statusId)
status_df = pd.read_csv(KAGGLE_DIR / 'status.csv', low_memory=False)
print(f"  ✓ Loaded status.csv: {len(status_df)} status types")

# Create status lookup dictionary: Status text -> statusId
status_lookup = dict(zip(status_df['status'].str.strip(), status_df['statusId']))
print(f"  ✓ Created status lookup dictionary: {len(status_lookup)} mappings")



Loading lookup tables...
  ✓ Loaded circuits.csv: 77 circuits
  ✓ Loaded drivers.csv: 861 drivers
  ✓ Loaded status.csv: 139 status types
  ✓ Created status lookup dictionary: 139 mappings


In [62]:
# Helper Functions

def ensure_driver_ids(drivers_df, fastf1_results):
    """
    Ensure every FastF1 driver code (Session == 'R') has a driverId.
    - If duplicate codes exist in drivers.csv, keep the max driverId.
    - If a code is missing, create a new row with driverId = max+1.
    Returns updated drivers_df and driver_code_to_id.
    """
    df = drivers_df.copy()

    df['code_norm'] = df['code'].astype(str).str.strip().str.upper()
    df['driverId_num'] = pd.to_numeric(df['driverId'], errors='coerce')

    # Dedup by code -> keep max driverId
    dedup = (
        df.dropna(subset=['code_norm', 'driverId_num'])
        .sort_values('driverId_num')
        .groupby('code_norm', as_index=False)
        .tail(1)
    )

    driver_code_to_id = dict(zip(dedup['code_norm'], dedup['driverId_num'].astype(int)))

    # FastF1 codes (race session only)
    fastf1_codes = set(
        fastf1_results[fastf1_results['Session'] == 'R']['Abbreviation']
        .dropna().astype(str).str.strip().str.upper().tolist()
    )

    missing_codes = sorted(list(fastf1_codes - set(driver_code_to_id.keys())))
    if missing_codes:
        next_id = int(dedup['driverId_num'].max()) + 1 if len(dedup) > 0 else 1
        new_rows = []
        for code in missing_codes:
            new_rows.append({
                'driverId': next_id,
                'driverRef': code.lower(),
                'number': pd.NA,
                'code': code,
                'forename': pd.NA,
                'surname': pd.NA,
                'dob': pd.NA,
                'nationality': pd.NA,
                'url': pd.NA
            })
            next_id += 1

        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

        # Rebuild map after adding new rows
        df['code_norm'] = df['code'].astype(str).str.strip().str.upper()
        df['driverId_num'] = pd.to_numeric(df['driverId'], errors='coerce')
        dedup = (
            df.dropna(subset=['code_norm', 'driverId_num'])
            .sort_values('driverId_num')
            .groupby('code_norm', as_index=False)
            .tail(1)
        )
        driver_code_to_id = dict(zip(dedup['code_norm'], dedup['driverId_num'].astype(int)))

    return df, driver_code_to_id

def normalize_event_name(name) -> str:
    """Normalize FastF1 event names for schedule/lookup joins."""
    if pd.isna(name):
        return ""
    s = str(name)
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\u00a0", " ").strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


def infer_round_date_from_laps(race_df, laps_df) -> pd.DataFrame:
    """
    Infer championship round order + race date using FastF1 lap timestamps.

    This is a FastF1-only fallback when ALL_EVENT_SCHEDULE.csv does not contain
    the target season (common if notebook 00 was run for a single year and
    overwrote ALL_EVENT_SCHEDULE.csv).
    """
    if len(race_df) == 0 or laps_df is None or len(laps_df) == 0:
        return pd.DataFrame(columns=["Year", "EventName_norm", "round", "date"])

    laps = laps_df.copy()
    if "LapStartDate" not in laps.columns or "Year" not in laps.columns or "Event" not in laps.columns:
        return pd.DataFrame(columns=["Year", "EventName_norm", "round", "date"])

    laps["Year"] = pd.to_numeric(laps["Year"], errors="coerce")
    laps["EventName_norm"] = laps["Event"].map(normalize_event_name)
    laps["LapStartDate"] = pd.to_datetime(laps["LapStartDate"], errors="coerce")

    # Use all sessions to get a stable per-event earliest timestamp (practice still orders events correctly).
    per_event = (
        laps.dropna(subset=["Year", "EventName_norm"])
        .groupby(["Year", "EventName_norm"], as_index=False)
        .agg(first_ts=("LapStartDate", "min"))
    )

    per_event = per_event.dropna(subset=["first_ts"])
    per_event["round"] = per_event.groupby("Year")["first_ts"].rank(method="dense", ascending=True).astype("Int64")
    per_event["date"] = per_event["first_ts"].dt.normalize()
    return per_event[["Year", "EventName_norm", "round", "date"]]


def build_event_schedule_mapping(race_df, event_schedule_df, laps_df) -> pd.DataFrame:
    """
    Build a (Year, EventName_norm) -> (round, date) table.

    Primary source: ALL_EVENT_SCHEDULE.csv (FastF1 extraction).
    Secondary source (FastF1-only): inferred ordering from ALL_LAPS_* timestamps.
    """
    years = sorted(pd.to_numeric(race_df["Year"], errors="coerce").dropna().unique().tolist())
    years = [int(y) for y in years]

    mapped_parts = []
    for y in years:
        race_y = race_df.loc[pd.to_numeric(race_df["Year"], errors="coerce") == y].copy()
        race_events = sorted({e for e in race_y["Event"].map(normalize_event_name).tolist() if e})

        sched_y = event_schedule_df[event_schedule_df["Year"] == y].copy() if len(event_schedule_df) else pd.DataFrame()

        sched_part = pd.DataFrame(columns=["Year", "EventName_norm", "round", "date"])
        if len(sched_y) > 0:
            sched_y["EventName_norm"] = sched_y["EventName"].map(normalize_event_name)
            sched_y["round"] = pd.to_numeric(sched_y["RoundNumber"], errors="coerce").astype("Int64")
            sched_y["date"] = pd.to_datetime(sched_y["EventDate"], errors="coerce").dt.normalize()
            sched_part = sched_y[["Year", "EventName_norm", "round", "date"]].dropna(subset=["EventName_norm", "round"])

        sched_events = sorted(set(sched_part["EventName_norm"].tolist())) if len(sched_part) else []
        missing_events = [e for e in race_events if e not in set(sched_events)]

        inferred_part = pd.DataFrame(columns=["Year", "EventName_norm", "round", "date"])
        if (len(sched_part) == 0 or len(missing_events) > 0) and laps_df is not None and len(laps_df) > 0:
            inferred_all = infer_round_date_from_laps(race_y, laps_df)
            inferred_all = inferred_all[inferred_all["Year"] == y]

            if len(sched_part) == 0:
                inferred_part = inferred_all
            else:
                inferred_part = inferred_all[inferred_all["EventName_norm"].isin(missing_events)]

        year_parts = []
        if len(sched_part):
            year_parts.append(sched_part)
        if len(inferred_part):
            year_parts.append(inferred_part)

        if year_parts:
            year_map = pd.concat(year_parts, ignore_index=True)
            # Prefer schedule rows over inferred rows when duplicates exist for the same event.
            year_map = year_map.drop_duplicates(subset=["Year", "EventName_norm"], keep="first")
            mapped_parts.append(year_map)

    if not mapped_parts:
        return pd.DataFrame(columns=["Year", "EventName_norm", "round", "date"])

    mapping = pd.concat(mapped_parts, ignore_index=True)
    mapping = mapping.dropna(subset=["Year", "EventName_norm", "round"])
    mapping["Year"] = pd.to_numeric(mapping["Year"], errors="coerce").astype(int)
    mapping["round"] = pd.to_numeric(mapping["round"], errors="coerce").astype(int)
    return mapping


def map_event_schedule_info(race_df, event_schedule_df, laps_df=None):
    """Attach `round` + `date` using FastF1 schedule data (no Kaggle)."""
    if len(race_df) == 0:
        return race_df

    race_df = race_df.copy()
    mapping = build_event_schedule_mapping(race_df, event_schedule_df, laps_df)

    race_df["EventName_norm"] = race_df["Event"].map(normalize_event_name)
    race_df["Year_int"] = pd.to_numeric(race_df["Year"], errors="coerce").astype(int)

    mapping_small = mapping.rename(columns={"Year": "Year_int"})
    race_df = race_df.merge(mapping_small, on=["Year_int", "EventName_norm"], how="left", validate="m:1")

    # Coerce to nullable integers where possible
    race_df["round"] = pd.to_numeric(race_df["round"], errors="coerce").astype("Int64")
    race_df["date"] = pd.to_datetime(race_df["date"], errors="coerce")

    race_df = race_df.drop(columns=["EventName_norm", "Year_int"], errors="ignore")
    return race_df

def parse_time_to_ms(val):
    """Parse time string to pandas Timedelta (handles FastF1 + master formats)."""
    if pd.isna(val):
        return pd.NaT
    s = str(val).strip()
    if s in ["", "\\N", "NaT", "None", "nan"]:
        return pd.NaT

    s = s.replace("\u202f", "").replace("\xa0", "").strip()
    if s.lower().endswith("lap"):
        return pd.NaT

    # Strip "0 days" if present
    if "days" in s:
        s = s.split("days", 1)[1].strip()

    # Normalize to HH:MM:SS.mmm for pd.to_timedelta
    if s.startswith("+"):
        s = s[1:].strip()
    if re.match(r"^\d+:\d{2}\.\d+$", s) or re.match(r"^\d+:\d{2}\.\d{3,6}$", s):
        s = "00:" + s
    elif re.match(r"^\d+:\d{2}:\d{2}$", s):
        s = s + ".000"
    elif re.match(r"^\d+\.\d+$", s):
        s = "00:00:" + s

    try:
        return pd.to_timedelta(s).round("1ms")
    except Exception:
        return pd.NaT


def time_to_milliseconds(time_val):
    """Convert time value to milliseconds."""
    td = parse_time_to_ms(time_val)
    if pd.isna(td):
        return pd.NA
    return int(round(td.total_seconds() * 1000))


def format_timedelta_to_time_str(td):
    """Format Timedelta to H:MM:SS.mmm or M:SS.mmm."""
    if pd.isna(td):
        return pd.NA
    total_seconds = td.total_seconds()
    hours = int(total_seconds // 3600)
    minutes = int((total_seconds % 3600) // 60)
    seconds = total_seconds % 60
    if hours > 0:
        return f"{hours}:{minutes:02d}:{seconds:06.3f}"
    return f"{minutes}:{seconds:06.3f}"


def normalize_quali_time(val):
    """Convert FastF1 timedelta string to MM:SS.mmm (or NA)."""
    td = parse_time_to_ms(val)
    if pd.isna(td):
        return pd.NA
    total_seconds = td.total_seconds()
    minutes = int(total_seconds // 60)
    seconds = total_seconds % 60
    return f"{minutes}:{seconds:06.3f}"


def convert_gap_times_to_absolute(race_df):
    """
    Convert FastF1 race gap times (small timedeltas) to absolute times.
    Gap definition: timedelta < 10 minutes AND position != 1.
    """
    if len(race_df) == 0:
        return race_df

    race_df = race_df.copy()
    race_df["_time_td"] = race_df["time"].apply(parse_time_to_ms)
    race_df["_pos_num"] = pd.to_numeric(race_df["position"], errors="coerce")

    gap_cutoff = pd.Timedelta(minutes=10)

    # Leader absolute time per (Year, Event)
    leader_td = {}
    for (year, event), grp in race_df.groupby(["Year", "Event"]):
        leader = grp[(grp["_pos_num"] == 1) & (grp["_time_td"].notna())]
        if len(leader) > 0:
            leader_td[(year, event)] = leader.iloc[0]["_time_td"]
        else:
            # Fallback: max timedelta in group (likely absolute)
            candidates = grp["_time_td"].dropna()
            if len(candidates) > 0:
                leader_td[(year, event)] = candidates.max()

    def to_absolute_td(row):
        td = row["_time_td"]
        if pd.isna(td):
            return pd.NaT
        base = leader_td.get((row["Year"], row["Event"]))
        if base is None:
            return td
        is_gap = (td < gap_cutoff) and (row["_pos_num"] != 1)
        return base + td if is_gap else td

    race_df["_time_abs_td"] = race_df.apply(to_absolute_td, axis=1)

    # Format ALL times to match master format
    race_df["time"] = race_df["_time_abs_td"].apply(format_timedelta_to_time_str)

    race_df = race_df.drop(columns=["_time_td", "_time_abs_td", "_pos_num"], errors="ignore")
    return race_df

def categorize_status_from_statusid(statusId):
    """Categorize statusId into simplified categories (same as master dataset)."""
    if pd.isna(statusId):
        return "Unknown"
    try:
        statusId = int(statusId)
    except (ValueError, TypeError):
        return "Unknown"
    
    if statusId == 1:  # Finished
        return "Finished"
    elif statusId in [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 45, 50, 53, 55, 58, 88, 
                      111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 122, 123, 124, 125, 127, 133, 134]:
        return "Finished_Lapped"
    elif statusId == 2:  # Disqualified
        return "Disqualified"
    elif statusId == 62:  # Not classified
        return "Not_Classified"
    else:
        return "DNF"

def map_fastf1_status_to_statusid(status_text):
    """Map FastF1 Status text to statusId using status.csv lookup."""
    if pd.isna(status_text):
        return pd.NA
    status_text = str(status_text).strip()
    return status_lookup.get(status_text, pd.NA)


# Sprint-only Ergast corrections when FastF1 status text differs from Kaggle.
SPRINT_STATUS_TEXT_OVERRIDES = {
    "Exhaust": 43,
}

SPRINT_STATUS_DRIVER_OVERRIDES = {
    (2024, "São Paulo Grand Prix", "HUL"): 43,
}


def map_sprint_status_to_statusid(row):
    """Map sprint Status to Ergast statusId with small override map."""
    year = int(row["Year"])
    event = str(row["Event"]).strip()
    code = str(row["code"]).strip().upper()
    driver_key = (year, event, code)
    if driver_key in SPRINT_STATUS_DRIVER_OVERRIDES:
        return SPRINT_STATUS_DRIVER_OVERRIDES[driver_key]

    status_text = row.get("status_text")
    if pd.notna(status_text):
        text = str(status_text).strip()
        if text in SPRINT_STATUS_TEXT_OVERRIDES:
            return SPRINT_STATUS_TEXT_OVERRIDES[text]

    return map_fastf1_status_to_statusid(status_text)


def apply_ergast_lapped_rules(race_df):
    """
    Apply Ergast/Kaggle rules after time conversion:
    - +N Lap finishers (statusId 11-19) have no race time / milliseconds
    - DNFs (no classified position) also have no time / milliseconds
    """
    if len(race_df) == 0:
        return race_df

    race_df = race_df.copy()
    winners = (
        race_df.loc[race_df["position"] == 1, ["Year", "Event", "laps"]]
        .drop_duplicates()
        .rename(columns={"laps": "_winner_laps"})
    )
    race_df = race_df.merge(winners, on=["Year", "Event"], how="left")
    laps_down = race_df["_winner_laps"] - race_df["laps"]

    finished = race_df["position"].notna()
    lapped = finished & laps_down.between(1, 9)
    race_df.loc[lapped, "statusId"] = (10 + laps_down[lapped]).astype("Int64")

    no_time = lapped | race_df["position"].isna()
    race_df.loc[no_time, "time"] = pd.NA
    race_df.loc[no_time, "milliseconds"] = pd.NA

    race_df = race_df.drop(columns=["_winner_laps"], errors="ignore")
    race_df["status_category"] = race_df["statusId"].apply(categorize_status_from_statusid)
    return race_df


def compute_ergast_rank(race_df):
    """Ergast rank: fastest-lap order within event; '0.0' when no fastest lap."""
    if len(race_df) == 0:
        return race_df

    race_df = race_df.copy()
    grp_keys = (
        ["year", "round"]
        if "year" in race_df.columns and "round" in race_df.columns
        else ["Year", "Event"]
    )
    race_df["rank"] = "0.0"
    mask = race_df["fastestLapTime"].notna()
    if not mask.any():
        return race_df

    race_df["_flt_td"] = pd.NA
    race_df.loc[mask, "_flt_td"] = race_df.loc[mask, "fastestLapTime"].apply(parse_time_to_ms)
    rankable = mask & race_df["_flt_td"].notna()
    if rankable.any():
        rk = race_df.loc[rankable].groupby(grp_keys)["_flt_td"].rank(method="min", ascending=True)
        race_df.loc[rankable, "rank"] = rk.map(lambda x: f"{float(x):.1f}")
    race_df = race_df.drop(columns=["_flt_td"], errors="ignore")
    return race_df

print("✓ Helper functions defined")

✓ Helper functions defined


## Load FastF1 Data (Year-Specific)


In [63]:
def load_fastf1_year(year):
    """
    Load FastF1 data for a specific year.
    
    Returns:
        dict: {
            'results': DataFrame (all sessions),
            'laps': DataFrame (all sessions),
            'telemetry': DataFrame (all sessions) - note: very large, use chunked reading for processing
        }
    """
    print(f"Loading FastF1 data for {year}...")
    
    results_file = FASTF1_DIR / f'ALL_RESULTS_{year}.csv'
    laps_file = FASTF1_DIR / f'ALL_LAPS_{year}.csv'
    telemetry_file = FASTF1_DIR / f'ALL_TELEMETRY_{year}.csv'
    
    data = {}
    
    # Load RESULTS
    if results_file.exists():
        results = pd.read_csv(results_file, low_memory=False)
        print(f"  ✓ RESULTS: {len(results):,} rows, {results['Session'].nunique()} sessions")
        data['results'] = results
    else:
        print(f"  ⚠ RESULTS file not found: {results_file.name}")
        data['results'] = pd.DataFrame()
    
    # Load LAPS
    if laps_file.exists():
        laps = pd.read_csv(laps_file, low_memory=False)
        print(f"  ✓ LAPS: {len(laps):,} rows, {laps['Session'].nunique()} sessions")
        data['laps'] = laps
    else:
        print(f"  ⚠ LAPS file not found: {laps_file.name}")
        data['laps'] = pd.DataFrame()
    
    # Note: Telemetry files are very large (5-6GB), we'll load them in chunks when needed
    # For now, just check if file exists
    if telemetry_file.exists():
        file_size_mb = telemetry_file.stat().st_size / (1024 * 1024)
        print(f"  ✓ TELEMETRY file exists: {file_size_mb:.1f} MB (will load in chunks when needed)")
        data['telemetry_file'] = telemetry_file
    else:
        print(f"  ⚠ TELEMETRY file not found: {telemetry_file.name}")
        data['telemetry_file'] = None
    
    return data

# Test loading 2024 data
print("Testing data loading for 2024:")
fastf1_2024 = load_fastf1_year(2024)
print()


Testing data loading for 2024:
Loading FastF1 data for 2024...
  ✓ RESULTS: 2,277 rows, 6 sessions
  ✓ LAPS: 62,690 rows, 6 sessions
  ⚠ TELEMETRY file not found: ALL_TELEMETRY_2024.csv



## Core Race Data Extraction


In [64]:
def extract_race_data(fastf1_data, year):
    """
    Extract core race data from FastF1 RESULTS.
    
    Returns DataFrame with one row per (Year, Event, Driver) for race session.
    """
    results = fastf1_data['results']
    
    if len(results) == 0:
        return pd.DataFrame()
    
    # Filter to Race session
    race_results = results[results['Session'] == 'R'].copy()
    
    if len(race_results) == 0:
        print(f"  ⚠ No race session data found for {year}")
        return pd.DataFrame()
    
    print(f"Extracting race data from {len(race_results)} race results...")
    
    # Extract core columns
    race_df = race_results[[
        'Year', 'Event', 'Abbreviation', 'DriverNumber',
        'GridPosition', 'Position', 'Points', 'Laps', 'Time', 'Status', 'TeamName'
    ]].copy()
    
    # Rename columns to match master structure
    race_df = race_df.rename(columns={
        'Abbreviation': 'code',
        'GridPosition': 'grid',
        'Position': 'position',
        'Points': 'points',
        'Laps': 'laps',
        'Time': 'time',
        'Status': 'status_text'
    })
    
    # Handle position: convert 'R' (retired) to NaN, numeric positions to float
    def parse_position(pos):
        if pd.isna(pos):
            return pd.NA
        pos_str = str(pos).strip()
        if pos_str.upper() == 'R' or pos_str == '':
            return pd.NA
        try:
            return float(pos_str)
        except ValueError:
            return pd.NA
    
    race_df['position'] = race_df['position'].apply(parse_position)
    
    # Convert grid to numeric
    race_df['grid'] = pd.to_numeric(race_df['grid'], errors='coerce')
    
    # Convert points and laps to numeric
    race_df['points'] = pd.to_numeric(race_df['points'], errors='coerce')
    race_df['laps'] = pd.to_numeric(race_df['laps'], errors='coerce')
    
    # Map status text to statusId
    race_df['statusId'] = race_df['status_text'].apply(map_fastf1_status_to_statusid)
    race_df['DriverNumber'] = pd.to_numeric(race_df['DriverNumber'], errors='coerce').astype('Int64')
    
    # Convert gap times to absolute times before converting to milliseconds
    race_df = convert_gap_times_to_absolute(race_df)
    
    # Convert time to milliseconds (now all times are absolute)
    race_df['milliseconds'] = race_df['time'].apply(time_to_milliseconds)

    # Ergast: lapped finishers and DNFs have no time/milliseconds; +N Lap statusIds
    race_df = apply_ergast_lapped_rules(race_df)
    
    # Extract qualifying times (from Session == 'Q')
    quali_results = results[results['Session'] == 'Q'].copy()
    if len(quali_results) > 0:
        quali_df = quali_results[['Year', 'Event', 'Abbreviation', 'Q1', 'Q2', 'Q3']].copy()
        quali_df = quali_df.rename(columns={'Abbreviation': 'code'})
        
        # Merge qualifying times
        race_df = race_df.merge(
            quali_df,
            on=['Year', 'Event', 'code'],
            how='left',
            suffixes=('', '_quali')
        )
        race_df['q1'] = race_df['Q1'].apply(normalize_quali_time)
        race_df['q2'] = race_df['Q2'].apply(normalize_quali_time)
        race_df['q3'] = race_df['Q3'].apply(normalize_quali_time)
        race_df = race_df.drop(columns=['Q1', 'Q2', 'Q3'])
    else:
        race_df['q1'] = pd.NA
        race_df['q2'] = pd.NA
        race_df['q3'] = pd.NA
    
    # Add year column (already have Year, but ensure consistency)
    race_df['year'] = race_df['Year']
    
    # Standardize code to uppercase
    race_df['code'] = race_df['code'].astype(str).str.strip().str.upper()
    
    # Ensure driverIds exist for all FastF1 codes (race session)
    global drivers_df, driver_code_to_id
    drivers_df, driver_code_to_id = ensure_driver_ids(drivers_df, results)
    
    # Map driver code to driverId
    race_df['driverId'] = race_df['code'].map(driver_code_to_id)
    
    print(f"  ✓ Extracted {len(race_df)} race entries")
    print(f"  ✓ Unique events: {race_df['Event'].nunique()}")
    print(f"  ✓ Unique drivers: {race_df['code'].nunique()}")
    
    return race_df

# Test extraction for 2024
print("Testing race data extraction for 2024:")
race_data_2024 = extract_race_data(fastf1_2024, 2024)
print(f"\nSample race data:")
print(race_data_2024[['year', 'Event', 'code', 'grid', 'position', 'points', 'laps', 'statusId', 'status_category']].head(10))


Testing race data extraction for 2024:
Extracting race data from 479 race results...
  ✓ Extracted 479 race entries
  ✓ Unique events: 24
  ✓ Unique drivers: 24

Sample race data:
   year               Event code  grid  position  points  laps statusId  \
0  2024  Bahrain Grand Prix  VER   1.0       1.0    26.0  57.0        1   
1  2024  Bahrain Grand Prix  PER   5.0       2.0    18.0  57.0        1   
2  2024  Bahrain Grand Prix  SAI   4.0       3.0    15.0  57.0        1   
3  2024  Bahrain Grand Prix  LEC   2.0       4.0    12.0  57.0        1   
4  2024  Bahrain Grand Prix  RUS   3.0       5.0    10.0  57.0        1   
5  2024  Bahrain Grand Prix  NOR   7.0       6.0     8.0  57.0        1   
6  2024  Bahrain Grand Prix  HAM   9.0       7.0     6.0  57.0        1   
7  2024  Bahrain Grand Prix  PIA   8.0       8.0     4.0  57.0        1   
8  2024  Bahrain Grand Prix  ALO   6.0       9.0     2.0  57.0        1   
9  2024  Bahrain Grand Prix  STR  12.0      10.0     1.0  57.0        

In [65]:
# FastF1 `Event` strings -> Kaggle `circuits.circuitRef`
#
# Rounds/dates must NOT come from Kaggle `races.csv` in this notebook; those are
# sourced from FastF1 `ALL_EVENT_SCHEDULE.csv` (with a FastF1-only laps fallback).
FASTF1_EVENT_TO_CIRCUITREF = {
    "70th Anniversary Grand Prix": "silverstone",
    "Abu Dhabi Grand Prix": "yas_marina",
    "Australian Grand Prix": "albert_park",
    "Austrian Grand Prix": "red_bull_ring",
    "Azerbaijan Grand Prix": "baku",
    "Bahrain Grand Prix": "bahrain",
    "Belgian Grand Prix": "spa",
    "Brazilian Grand Prix": "interlagos",
    "British Grand Prix": "silverstone",
    "Canadian Grand Prix": "villeneuve",
    "Chinese Grand Prix": "shanghai",
    "Dutch Grand Prix": "zandvoort",
    "Eifel Grand Prix": "nurburgring",
    "Emilia Romagna Grand Prix": "imola",
    "French Grand Prix": "ricard",
    "German Grand Prix": "hockenheimring",
    "Hungarian Grand Prix": "hungaroring",
    "Italian Grand Prix": "monza",
    "Japanese Grand Prix": "suzuka",
    "Las Vegas Grand Prix": "vegas",
    "Mexican Grand Prix": "rodriguez",
    "Mexico City Grand Prix": "rodriguez",
    "Miami Grand Prix": "miami",
    "Monaco Grand Prix": "monaco",
    "Portuguese Grand Prix": "portimao",
    "Qatar Grand Prix": "losail",
    "Russian Grand Prix": "sochi",
    "Saudi Arabian Grand Prix": "jeddah",
    "Singapore Grand Prix": "marina_bay",
    "Spanish Grand Prix": "catalunya",
    "Styrian Grand Prix": "red_bull_ring",
    "São Paulo Grand Prix": "interlagos",
    "Turkish Grand Prix": "istanbul",
    "Tuscan Grand Prix": "mugello",
    "United States Grand Prix": "americas",
}


def map_circuit_info(race_df, circuits_df):
    """Map circuitId + lat/lng from FastF1 Event -> circuits.csv (no races.csv)."""
    if len(race_df) == 0:
        return race_df

    print("Mapping circuit information...")

    circuits = circuits_df.copy()
    circuits["circuitId"] = pd.to_numeric(circuits["circuitId"], errors="coerce").astype("Int64")
    ref_to_id = dict(
        zip(
            circuits["circuitRef"].astype(str).str.strip().str.lower(),
            circuits["circuitId"],
        )
    )

    event_to_circuit_ref = {
        normalize_event_name(k): str(v).strip().lower() for k, v in FASTF1_EVENT_TO_CIRCUITREF.items()
    }

    circuit_coords = dict(zip(circuits["circuitId"], zip(circuits["lat"], circuits["lng"])))

    race_df = race_df.copy()
    race_df["name"] = race_df["Event"]
    race_df["EventName_norm"] = race_df["Event"].map(normalize_event_name)

    circuit_refs = race_df["EventName_norm"].map(event_to_circuit_ref)
    race_df["circuitId"] = circuit_refs.map(lambda ref: ref_to_id.get(ref, pd.NA) if pd.notna(ref) and ref != "" else pd.NA)

    race_df["lat"] = race_df["circuitId"].map(lambda x: circuit_coords.get(x, (pd.NA, pd.NA))[0] if pd.notna(x) else pd.NA)
    race_df["lng"] = race_df["circuitId"].map(lambda x: circuit_coords.get(x, (pd.NA, pd.NA))[1] if pd.notna(x) else pd.NA)

    race_df = race_df.drop(columns=["EventName_norm"], errors="ignore")

    unmapped = race_df[race_df["circuitId"].isna()]
    if len(unmapped) > 0:
        print(f"  ⚠ Warning: {len(unmapped)} rows could not be mapped to circuitId")
        print(f"    Unmapped events: {unmapped['Event'].unique().tolist()}")
    else:
        print(f"  ✓ All {len(race_df)} rows mapped to circuits")

    return race_df


schedule_path = FASTF1_DIR / "ALL_EVENT_SCHEDULE.csv"
if schedule_path.exists():
    event_schedule_df = pd.read_csv(schedule_path, low_memory=False)
    print(f"Loaded FastF1 schedule: {schedule_path.name} ({len(event_schedule_df)} rows)")
else:
    event_schedule_df = pd.DataFrame()
    print(f"⚠ FastF1 schedule not found at {schedule_path} (rounds will fall back to lap timestamps if possible)")

# Test circuit + schedule mapping for 2024 (matches reproduce_fastf1_year order)
print("\nTesting circuit mapping for 2024:")
race_data_2024 = map_circuit_info(race_data_2024, circuits_df)
# round/date come from FastF1 schedule (not map_circuit_info); align with pipeline before printing
race_data_2024 = map_event_schedule_info(
    race_data_2024, event_schedule_df, laps_df=fastf1_2024.get("laps")
)
print("\nSample with circuit + schedule columns:")
print(race_data_2024[["year", "Event", "circuitId", "round", "date", "lat", "lng"]].head(10))


Loaded FastF1 schedule: ALL_EVENT_SCHEDULE.csv (195 rows)

Testing circuit mapping for 2024:
Mapping circuit information...
  ✓ All 479 rows mapped to circuits

Sample with circuit + schedule columns:
   year               Event  circuitId  round       date      lat      lng
0  2024  Bahrain Grand Prix          3      1 2024-03-02  26.0325  50.5106
1  2024  Bahrain Grand Prix          3      1 2024-03-02  26.0325  50.5106
2  2024  Bahrain Grand Prix          3      1 2024-03-02  26.0325  50.5106
3  2024  Bahrain Grand Prix          3      1 2024-03-02  26.0325  50.5106
4  2024  Bahrain Grand Prix          3      1 2024-03-02  26.0325  50.5106
5  2024  Bahrain Grand Prix          3      1 2024-03-02  26.0325  50.5106
6  2024  Bahrain Grand Prix          3      1 2024-03-02  26.0325  50.5106
7  2024  Bahrain Grand Prix          3      1 2024-03-02  26.0325  50.5106
8  2024  Bahrain Grand Prix          3      1 2024-03-02  26.0325  50.5106
9  2024  Bahrain Grand Prix          3      1 202

## Fastest Lap Calculations (from LAPS data)


In [66]:
def calculate_fastest_lap_features(race_df, laps_df):
    """
    Calculate fastestLap and fastestLapTime from ALL_LAPS data.
    
    For each (Year, Event, Driver):
    - Filter LAPS to Session == 'R', that driver
    - Find minimum LapTime (exclude 0, NaN, deleted laps)
    - fastestLap → lap number of fastest lap
    - fastestLapTime → time of fastest lap
    """
    if len(race_df) == 0 or len(laps_df) == 0:
        race_df['fastestLap'] = pd.NA
        race_df['fastestLapTime'] = pd.NA
        race_df['rank'] = '0.0'
        return race_df
    
    print("Calculating fastest lap features from LAPS data...")
    
    # Filter to race session only
    race_laps = laps_df[laps_df['Session'] == 'R'].copy()
    
    if len(race_laps) == 0:
        print("  ⚠ No race lap data found")
        race_df['fastestLap'] = pd.NA
        race_df['fastestLapTime'] = pd.NA
        race_df['rank'] = pd.NA
        return race_df
    
    # Parse lap times and filter valid laps
    def parse_lap_time(lap_time):
        """Parse lap time string to timedelta."""
        if pd.isna(lap_time):
            return pd.NaT
        td = parse_time_to_ms(lap_time)
        return td
    
    race_laps['LapTime_parsed'] = race_laps['LapTime'].apply(parse_lap_time)
    
    # Filter out invalid laps: NaN, 0, deleted laps
    valid_laps = race_laps[
        race_laps['LapTime_parsed'].notna() &
        (race_laps['LapTime_parsed'] > pd.Timedelta(0)) &
        (race_laps.get('Deleted', pd.Series([False] * len(race_laps))) != True)
    ].copy()
    
    if len(valid_laps) == 0:
        print("  ⚠ No valid lap times found")
        race_df['fastestLap'] = pd.NA
        race_df['fastestLapTime'] = pd.NA
        race_df['rank'] = '0.0'
        return race_df
    
    if 'TrackStatus' in valid_laps.columns:
        valid_laps = valid_laps[valid_laps['TrackStatus'] == 1]
    if 'IsAccurate' in valid_laps.columns:
        valid_laps = valid_laps[valid_laps['IsAccurate'] == True]

    valid_laps['DriverNumber'] = pd.to_numeric(valid_laps['DriverNumber'], errors='coerce').astype('Int64')
    race_df['DriverNumber'] = pd.to_numeric(race_df['DriverNumber'], errors='coerce').astype('Int64')

    # Group by (Year, Event, DriverNumber) — reliable join key vs Driver string
    fastest_laps = valid_laps.loc[
        valid_laps.groupby(['Year', 'Event', 'DriverNumber'])['LapTime_parsed'].idxmin()
    ][['Year', 'Event', 'DriverNumber', 'LapNumber', 'LapTime']].copy()

    fastest_laps = fastest_laps.rename(columns={
        'LapNumber': 'fastestLap',
        'LapTime': 'fastestLapTime'
    })

    race_df = race_df.merge(
        fastest_laps,
        on=['Year', 'Event', 'DriverNumber'],
        how='left',
        suffixes=('', '_fastest')
    )

    race_df['fastestLapTime'] = race_df['fastestLapTime'].apply(normalize_quali_time)

    # Ergast: no fastest lap for unclassified finishers (DNF/DQ)
    no_fl = race_df['position'].isna()
    race_df.loc[no_fl, ['fastestLap', 'fastestLapTime']] = pd.NA

    race_df = compute_ergast_rank(race_df)
    
    matched = race_df['fastestLap'].notna().sum()
    print(f"  ✓ Found fastest lap for {matched}/{len(race_df)} drivers")
    
    return race_df

# Test fastest lap calculation for 2024
print("Testing fastest lap calculation for 2024:")
race_data_2024 = calculate_fastest_lap_features(race_data_2024, fastf1_2024['laps'])
print(f"\nSample with fastest lap:")
print(race_data_2024[['year', 'Event', 'code', 'fastestLap', 'fastestLapTime']].head(10))


Testing fastest lap calculation for 2024:
Calculating fastest lap features from LAPS data...
  ✓ Found fastest lap for 463/479 drivers

Sample with fastest lap:
   year               Event code  fastestLap fastestLapTime
0  2024  Bahrain Grand Prix  VER        39.0       1:32.608
1  2024  Bahrain Grand Prix  PER        40.0       1:34.364
2  2024  Bahrain Grand Prix  SAI        44.0       1:34.507
3  2024  Bahrain Grand Prix  LEC        36.0       1:34.090
4  2024  Bahrain Grand Prix  RUS        40.0       1:35.065
5  2024  Bahrain Grand Prix  NOR        35.0       1:34.476
6  2024  Bahrain Grand Prix  HAM        39.0       1:34.722
7  2024  Bahrain Grand Prix  PIA        39.0       1:34.983
8  2024  Bahrain Grand Prix  ALO        48.0       1:34.199
9  2024  Bahrain Grand Prix  STR        30.0       1:35.632


## Fastest Lap Speed (from TELEMETRY)

**Note**: This requires matching telemetry data to specific laps using LapStartTime/LapStartDate from LAPS data. 
For now, we'll use SpeedFL from LAPS data (speed at finish line) as a proxy, or calculate from telemetry if needed.


In [67]:
def calculate_fastest_lap_speed(race_df, laps_df):
    """
    Calculate fastestLapSpeed from LAPS data (using SpeedFL - speed at finish line).
    
    For a more accurate calculation, we would need to:
    1. Match telemetry data to specific laps using LapStartTime/LapStartDate
    2. Filter telemetry to that lap's time window
    3. Calculate average Speed for that lap
    
    For now, we use SpeedFL from the fastest lap in LAPS data as a proxy.
    """
    if len(race_df) == 0 or len(laps_df) == 0:
        race_df['fastestLapSpeed'] = pd.NA
        return race_df
    
    print("Calculating fastest lap speed from LAPS data...")
    
    # Filter to race session
    race_laps = laps_df[laps_df['Session'] == 'R'].copy()
    
    if len(race_laps) == 0:
        print("  ⚠ No race lap data found")
        race_df['fastestLapSpeed'] = pd.NA
        return race_df
    
    # Get fastest lap info for each driver (already calculated in race_df)
    # Match back to LAPS to get SpeedFL for that lap
    fastest_lap_speeds = []
    
    for _, row in race_df.iterrows():
        if pd.isna(row.get('fastestLap')):
            fastest_lap_speeds.append(pd.NA)
            continue
        
        # Find the fastest lap in LAPS data (join on DriverNumber, not Driver string)
        driver_laps = race_laps[
            (race_laps['Year'] == row['Year']) &
            (race_laps['Event'] == row['Event']) &
            (race_laps['DriverNumber'] == row['DriverNumber']) &
            (race_laps['LapNumber'] == row['fastestLap'])
        ]
        
        if len(driver_laps) > 0 and 'SpeedFL' in driver_laps.columns:
            speed = driver_laps.iloc[0]['SpeedFL']
            fastest_lap_speeds.append(speed if pd.notna(speed) else pd.NA)
        else:
            fastest_lap_speeds.append(pd.NA)
    
    race_df['fastestLapSpeed'] = fastest_lap_speeds
    
    matched = race_df['fastestLapSpeed'].notna().sum()
    print(f"  ✓ Found fastest lap speed for {matched}/{len(race_df)} drivers")
    
    return race_df

# Test fastest lap speed calculation for 2024
print("Testing fastest lap speed calculation for 2024:")
race_data_2024 = calculate_fastest_lap_speed(race_data_2024, fastf1_2024['laps'])
print(f"\nSample with fastest lap speed:")
print(race_data_2024[['year', 'Event', 'code', 'fastestLap', 'fastestLapSpeed']].head(10))


Testing fastest lap speed calculation for 2024:
Calculating fastest lap speed from LAPS data...
  ✓ Found fastest lap speed for 463/479 drivers

Sample with fastest lap speed:
   year               Event code  fastestLap fastestLapSpeed
0  2024  Bahrain Grand Prix  VER        39.0           281.0
1  2024  Bahrain Grand Prix  PER        40.0           281.0
2  2024  Bahrain Grand Prix  SAI        44.0           282.0
3  2024  Bahrain Grand Prix  LEC        36.0           281.0
4  2024  Bahrain Grand Prix  RUS        40.0           282.0
5  2024  Bahrain Grand Prix  NOR        35.0           287.0
6  2024  Bahrain Grand Prix  HAM        39.0           282.0
7  2024  Bahrain Grand Prix  PIA        39.0           289.0
8  2024  Bahrain Grand Prix  ALO        48.0           280.0
9  2024  Bahrain Grand Prix  STR        30.0           281.0


## Driver Age Mapping


In [68]:
def add_driver_age(race_df, drivers_df):
    """
    Map driver age from drivers.csv using code → driverId → dob lookup.
    Calculate driver_age = race date - dob.
    """
    if len(race_df) == 0:
        race_df['driver_age'] = pd.NA
        return race_df
    
    print("Calculating driver age...")
    
    # Create driver code -> dob lookup
    driver_dob = dict(zip(
        drivers_df['code'].astype(str).str.strip().str.upper(),
        pd.to_datetime(drivers_df['dob'])
    ))
    
    # Map dob to race_df
    race_df['driver_dob'] = race_df['code'].map(driver_dob)
    
    # Calculate age at race date
    race_df['driver_age'] = (race_df['date'] - race_df['driver_dob']).dt.days / 365.25
    
    # Clean up temporary column
    race_df = race_df.drop(columns=['driver_dob'])
    
    matched = race_df['driver_age'].notna().sum()
    print(f"  ✓ Calculated age for {matched}/{len(race_df)} drivers")
    
    return race_df

# Test driver age calculation for 2024
print("Testing driver age calculation for 2024:")
race_data_2024 = add_driver_age(race_data_2024, drivers_df)
print(f"\nSample with driver age:")
print(race_data_2024[['year', 'Event', 'code', 'driverId', 'date', 'driver_age']].head(10))


Testing driver age calculation for 2024:
Calculating driver age...
  ✓ Calculated age for 479/479 drivers

Sample with driver age:
   year               Event code  driverId       date  driver_age
0  2024  Bahrain Grand Prix  VER       830 2024-03-02   26.420260
1  2024  Bahrain Grand Prix  PER       815 2024-03-02   34.097194
2  2024  Bahrain Grand Prix  SAI       832 2024-03-02   29.500342
3  2024  Bahrain Grand Prix  LEC       844 2024-03-02   26.376454
4  2024  Bahrain Grand Prix  RUS       847 2024-03-02   26.042437
5  2024  Bahrain Grand Prix  NOR       846 2024-03-02   24.301164
6  2024  Bahrain Grand Prix  HAM         1 2024-03-02   39.148528
7  2024  Bahrain Grand Prix  PIA       857 2024-03-02   22.904860
8  2024  Bahrain Grand Prix  ALO         4 2024-03-02   42.592745
9  2024  Bahrain Grand Prix  STR       840 2024-03-02   25.341547


## Sprint Results Extraction

Sprint results are attached to the same row as race data (matched by Year, Event, Driver).


In [69]:
def _parse_finish_position(pos):
    if pd.isna(pos):
        return pd.NA
    pos_str = str(pos).strip()
    if pos_str.upper() in ("R", ""):
        return pd.NA
    try:
        return float(pos_str)
    except ValueError:
        return pd.NA


def fill_sprint_na_with_zeros(race_df, sprint_event_keys=None):
    """
    Match master sentinels on non-sprint weekends (0 / '0:00.000').
    On sprint weekends, keep NA for DNFs (do not coerce time to 0).
    """
    sprint_cols = [
        "sprint_results_grid", "sprint_results_positionOrder", "sprint_results_points",
        "sprint_results_laps", "sprint_results_time", "sprint_results_milliseconds",
        "sprint_results_fastestLap", "sprint_results_fastestLapTime", "sprint_results_statusId",
    ]
    if not any(c in race_df.columns for c in sprint_cols):
        return race_df

    race_df = race_df.copy()
    if sprint_event_keys is None:
        in_sprint = (
            race_df["sprint_results_positionOrder"].notna()
            if "sprint_results_positionOrder" in race_df.columns
            else pd.Series(False, index=race_df.index)
        )
    else:
        in_sprint = race_df.apply(
            lambda r: (r["Year"], r["Event"]) in sprint_event_keys,
            axis=1,
        )

    numeric_like = [
        "sprint_results_grid", "sprint_results_positionOrder", "sprint_results_points",
        "sprint_results_laps", "sprint_results_milliseconds", "sprint_results_fastestLap",
        "sprint_results_statusId",
    ]
    for col in numeric_like:
        if col not in race_df.columns:
            continue
        fill_mask = ~in_sprint
        race_df.loc[fill_mask, col] = pd.to_numeric(race_df.loc[fill_mask, col], errors="coerce").fillna(0)

    if "sprint_results_time" in race_df.columns:
        race_df.loc[~in_sprint, "sprint_results_time"] = "0:00.000"
        race_df.loc[~in_sprint, "sprint_results_milliseconds"] = 0

    if "sprint_results_fastestLapTime" in race_df.columns:
        race_df["sprint_results_fastestLapTime"] = race_df["sprint_results_fastestLapTime"].apply(normalize_quali_time)
        race_df.loc[~in_sprint & race_df["sprint_results_fastestLapTime"].isna(), "sprint_results_fastestLapTime"] = "0"

    return race_df


def extract_sprint_data(race_df, fastf1_data, year):
    """
    Sprint results aligned to Kaggle/master:
    - time / milliseconds / laps from FastF1 RESULTS (Sprint), with gap→absolute like race
    - fastest lap / fastest lap time from LAPS (Sprint), excluding deleted laps
    """
    results = fastf1_data["results"]
    laps = fastf1_data["laps"]

    sprint_cols = [
        "sprint_results_grid", "sprint_results_positionOrder", "sprint_results_points",
        "sprint_results_laps", "sprint_results_time", "sprint_results_milliseconds",
        "sprint_results_fastestLap", "sprint_results_fastestLapTime", "sprint_results_statusId",
    ]

    if len(results) == 0:
        for col in sprint_cols:
            race_df[col] = pd.NA
        return fill_sprint_na_with_zeros(race_df, sprint_event_keys=set())

    print("Extracting sprint results...")
    sprint_results = results[results["Session"] == "Sprint"].copy()

    if len(sprint_results) == 0:
        print("  ⚠ No sprint session data found")
        for col in sprint_cols:
            race_df[col] = pd.NA
        return fill_sprint_na_with_zeros(race_df, sprint_event_keys=set())

    sprint_event_keys = set(zip(sprint_results["Year"], sprint_results["Event"]))

    sprint_df = sprint_results[
        ["Year", "Event", "Abbreviation", "DriverNumber", "GridPosition", "Position", "Points", "Laps", "Time", "Status"]
    ].copy()

    sprint_df = sprint_df.rename(columns={
        "Abbreviation": "code",
        "GridPosition": "sprint_results_grid",
        "Position": "sprint_results_positionOrder",
        "Points": "sprint_results_points",
        "Laps": "sprint_results_laps",
        "Time": "time",
        "Status": "status_text",
    })
    sprint_df["code"] = sprint_df["code"].astype(str).str.strip().str.upper()
    sprint_df["DriverNumber"] = pd.to_numeric(sprint_df["DriverNumber"], errors="coerce").astype("Int64")
    sprint_df["sprint_results_positionOrder"] = sprint_df["sprint_results_positionOrder"].apply(_parse_finish_position)
    sprint_df["sprint_results_grid"] = pd.to_numeric(sprint_df["sprint_results_grid"], errors="coerce")
    sprint_df["sprint_results_points"] = pd.to_numeric(sprint_df["sprint_results_points"], errors="coerce")
    sprint_df["sprint_results_laps"] = pd.to_numeric(sprint_df["sprint_results_laps"], errors="coerce")
    sprint_df["sprint_results_statusId"] = sprint_df.apply(map_sprint_status_to_statusid, axis=1)
    sprint_df = sprint_df.drop(columns=["status_text"])

    # Official finish time (same pipeline as extract_race_data)
    tmp = sprint_df.rename(columns={"sprint_results_positionOrder": "position"})
    tmp = convert_gap_times_to_absolute(tmp)
    sprint_df["sprint_results_time"] = tmp["time"]
    sprint_df["sprint_results_milliseconds"] = sprint_df["sprint_results_time"].apply(time_to_milliseconds)

    # Fastest sprint lap from LAPS (not for total race time)
    sprint_laps = laps[laps["Session"] == "Sprint"].copy()
    if len(sprint_laps) > 0:
        sprint_laps["LapTime_parsed"] = sprint_laps["LapTime"].apply(parse_time_to_ms)
        deleted = sprint_laps.get("Deleted", pd.Series(False, index=sprint_laps.index))
        valid = sprint_laps[
            sprint_laps["LapTime_parsed"].notna()
            & (sprint_laps["LapTime_parsed"] > pd.Timedelta(0))
            & (deleted != True)
        ]
        if len(valid) > 0:
            valid["DriverNumber"] = pd.to_numeric(valid["DriverNumber"], errors="coerce").astype("Int64")
            fastest = valid.loc[
                valid.groupby(["Year", "Event", "DriverNumber"])["LapTime_parsed"].idxmin()
            ][["Year", "Event", "DriverNumber", "LapNumber", "LapTime"]].copy()
            fastest = fastest.rename(columns={
                "LapNumber": "sprint_results_fastestLap",
                "LapTime": "sprint_results_fastestLapTime",
            })
            sprint_df = sprint_df.merge(fastest, on=["Year", "Event", "DriverNumber"], how="left")

    race_df = race_df.merge(sprint_df, on=["Year", "Event", "code"], how="left", suffixes=("", "_sprint"))

    if "sprint_results_fastestLapTime" in race_df.columns:
        race_df["sprint_results_fastestLapTime"] = race_df["sprint_results_fastestLapTime"].apply(normalize_quali_time)

    matched = race_df["sprint_results_grid"].notna().sum()
    print(f"  ✓ Found sprint results for {matched}/{len(race_df)} drivers")
    print(f"  ✓ Sprint events this year: {len(sprint_event_keys)}")

    return fill_sprint_na_with_zeros(race_df, sprint_event_keys=sprint_event_keys)

# Test sprint extraction for 2024
print("Testing sprint results extraction for 2024:")
race_data_2024 = extract_sprint_data(race_data_2024, fastf1_2024, 2024)
print(f"\nSample with sprint results:")
sprint_cols = [col for col in race_data_2024.columns if 'sprint' in col.lower()]
if sprint_cols:
    print(race_data_2024[['year', 'Event', 'code'] + sprint_cols].head(10))
else:
    print("No sprint columns found")


Testing sprint results extraction for 2024:
Extracting sprint results...
  ✓ Found sprint results for 120/479 drivers
  ✓ Sprint events this year: 6

Sample with sprint results:
   year               Event code  DriverNumber_sprint  sprint_results_grid  \
0  2024  Bahrain Grand Prix  VER                 <NA>                  0.0   
1  2024  Bahrain Grand Prix  PER                 <NA>                  0.0   
2  2024  Bahrain Grand Prix  SAI                 <NA>                  0.0   
3  2024  Bahrain Grand Prix  LEC                 <NA>                  0.0   
4  2024  Bahrain Grand Prix  RUS                 <NA>                  0.0   
5  2024  Bahrain Grand Prix  NOR                 <NA>                  0.0   
6  2024  Bahrain Grand Prix  HAM                 <NA>                  0.0   
7  2024  Bahrain Grand Prix  PIA                 <NA>                  0.0   
8  2024  Bahrain Grand Prix  ALO                 <NA>                  0.0   
9  2024  Bahrain Grand Prix  STR          

In [70]:
import re

# Load constructors.csv to build TeamName -> constructorId lookup (no fallback)
constructors_df = pd.read_csv(KAGGLE_DIR / 'constructors.csv', low_memory=False)

TEAM_ALIASES = {
    # FastF1 → Kaggle constructor names
    "red bull racing": "red bull",
    "rb f1 team": "rb",
    "racing bulls": "rb",
    "alpha tauri": "alphatauri",
    "alphatauri": "alphatauri",
    "toro rosso": "toro rosso",
    "aston martin": "aston martin",
    "kick sauber": "sauber",
    "stake": "sauber",
    "sauber": "sauber",
    "alpine": "alpine f1 team",
    "haas f1 team": "haas f1 team",
    "williams": "williams",
    "mclaren": "mclaren",
    "ferrari": "ferrari",
    "mercedes": "mercedes",
}

def normalize_team_name(name):
    # Lowercase, remove quotes, collapse whitespace, strip punctuation
    s = str(name).strip().strip('"').strip("'").lower()
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    # Apply alias if known
    return TEAM_ALIASES.get(s, s)

teamname_to_constructorid = dict(zip(
    constructors_df['name'].apply(normalize_team_name),
    constructors_df['constructorId']
))

print(f"Created TeamName -> constructorId lookup: {len(teamname_to_constructorid)} mappings")

def get_constructor_id(row):
    team = normalize_team_name(row.get('TeamName', ''))
    return teamname_to_constructorid.get(team, pd.NA)

Created TeamName -> constructorId lookup: 212 mappings


## Standings Calculations


In [71]:
def calculate_standings(race_df):
    """
    Calculate driver and constructor standings (points, position) from cumulative points.
    PRE_RACE cumulative points use .shift(1) within each season; rows with no prior race
    in that championship (shift NA) are filled with 0 so PRE points are never missing.
    PRE_RACE positions remain NA on each entity's first round of the season (no meaningful rank).
    """
    if len(race_df) == 0:
        return race_df
    
    print("Calculating standings...")

    if "round" not in race_df.columns:
        raise ValueError("calculate_standings(): missing required column `round`.")

    round_non_null = int(pd.to_numeric(race_df["round"], errors="coerce").notna().sum())
    n = len(race_df)
    miss = n - round_non_null
    miss_frac = (miss / n) if n else 0.0

    # Hard guard: standings groupby keys require a real championship round.
    if miss_frac > 0.01:
        bad = race_df.loc[pd.to_numeric(race_df["round"], errors="coerce").isna(), ["year", "Event"]].drop_duplicates()
        sample = bad.head(25).to_string(index=False)
        raise ValueError(
            "calculate_standings(): `round` is mostly missing — cannot compute standings safely.\n"
            f"rows={n}, round_non_null={round_non_null}, missing={miss} ({miss_frac:.2%}).\n"
            "Fix FastF1 `ALL_EVENT_SCHEDULE.csv` coverage for this season (notebook 00 overwrites it per run),\n"
            "or ensure ALL_LAPS_* timestamps are available for lap-based round inference.\n"
            f"Unmapped (year, Event) sample:\n{sample}"
        )
    
    # Sort by year, round, date for proper temporal ordering
    race_df = race_df.sort_values(['year', 'round', 'date']).reset_index(drop=True)
    
    # Standings points basis: race points + sprint points (where available)
    race_df['points'] = pd.to_numeric(race_df['points'], errors='coerce').fillna(0)
    if 'sprint_results_points' in race_df.columns:
        race_df['sprint_results_points'] = pd.to_numeric(race_df['sprint_results_points'], errors='coerce').fillna(0)
    else:
        race_df['sprint_results_points'] = 0

    race_df['standings_points_delta'] = race_df['points'] + race_df['sprint_results_points']

    # Driver standings points: cumulative sum per driver (season reset)
    race_df['driver_standings_points'] = race_df.groupby(['year', 'driverId'])['standings_points_delta'].cumsum()
    
    # Driver standings position: rank on unique (year, round, driverId), then map back.
    driver_round_points = (
        race_df.groupby(['year', 'round', 'driverId'], as_index=False)['driver_standings_points']
        .max()
    )
    driver_round_points['driver_standings_position'] = (
        driver_round_points.groupby(['year', 'round'])['driver_standings_points']
        .rank(method='min', ascending=False)
        .astype("Int64")
    )
    race_df = race_df.drop(columns=['driver_standings_position'], errors='ignore')
    race_df = race_df.merge(
        driver_round_points[['year', 'round', 'driverId', 'driver_standings_position']],
        on=['year', 'round', 'driverId'],
        how='left'
    )
    
    # Constructor standings points: constructor race points -> season cumsum (constructor-race granularity)
    constructor_points_keys = ['year', 'round', 'constructorId']
    if 'raceId' in race_df.columns:
        constructor_points_keys = ['year', 'round', 'raceId', 'constructorId']

    constructor_race_points = (
        race_df.groupby(constructor_points_keys, as_index=False)['standings_points_delta']
        .sum()
        .rename(columns={'standings_points_delta': 'constructor_race_points'})
    )

    sort_cols = [c for c in ['year', 'constructorId', 'round', 'raceId'] if c in constructor_race_points.columns]
    constructor_race_points = constructor_race_points.sort_values(sort_cols).reset_index(drop=True)

    constructor_race_points['constructor_standings_points'] = (
        constructor_race_points.groupby(['year', 'constructorId'])['constructor_race_points'].cumsum()
    )

    # Overwrite constructor standings points on the driver-row table
    race_df = race_df.drop(columns=['constructor_standings_points'], errors='ignore')
    race_df = race_df.merge(
        constructor_race_points[constructor_points_keys + ['constructor_standings_points']],
        on=constructor_points_keys,
        how='left'
    )

    # Constructor standings position: rank on unique constructor table, then map back.
    constructor_rank_cols = ['year', 'round', 'constructorId', 'constructor_standings_points']
    if 'raceId' in constructor_race_points.columns:
        constructor_rank_cols.insert(2, 'raceId')

    constructor_rank_table = constructor_race_points[constructor_rank_cols].copy()
    constructor_rank_table['constructor_standings_position'] = (
        constructor_rank_table.groupby(['year', 'round'])['constructor_standings_points']
        .rank(method='min', ascending=False)
        .astype("Int64")
    )

    race_df = race_df.drop(columns=['constructor_standings_position'], errors='ignore')
    race_df = race_df.merge(
        constructor_rank_table[[c for c in constructor_points_keys if c in constructor_rank_table.columns] + ['constructor_standings_position']],
        on=constructor_points_keys,
        how='left'
    )
    
    # PRE_RACE versions (season-scoped)
    race_df['driver_standings_points_PRE_RACE'] = (
        race_df.groupby(['year', 'driverId'])['driver_standings_points'].shift(1)
    )
    race_df['driver_standings_points_PRE_RACE'] = race_df['driver_standings_points_PRE_RACE'].fillna(0)

    # Driver PRE positions: rank on unique (year, round, driverId), then map back.
    driver_pre_points = (
        race_df.groupby(['year', 'round', 'driverId'], as_index=False)['driver_standings_points_PRE_RACE']
        .max()
    )
    driver_pre_points['driver_standings_position_PRE_RACE'] = (
        driver_pre_points.groupby(['year', 'round'])['driver_standings_points_PRE_RACE']
        .rank(method='min', ascending=False)
        .astype("Int64")
    )
    race_df = race_df.drop(columns=['driver_standings_position_PRE_RACE'], errors='ignore')
    race_df = race_df.merge(
        driver_pre_points[['year', 'round', 'driverId', 'driver_standings_position_PRE_RACE']],
        on=['year', 'round', 'driverId'],
        how='left'
    )

    # Ensure season opener stays missing for PRE_RACE positions
    first_round_by_driver = race_df.groupby(['year', 'driverId'])['round'].transform('min')
    race_df.loc[
        race_df['round'] == first_round_by_driver,
        'driver_standings_position_PRE_RACE'
    ] = pd.NA

    # Constructor PRE points from constructor-race granularity
    constructor_race_points['constructor_standings_points_PRE_RACE'] = (
        constructor_race_points.groupby(['year', 'constructorId'])['constructor_standings_points'].shift(1)
    )
    constructor_race_points['constructor_standings_points_PRE_RACE'] = (
        constructor_race_points['constructor_standings_points_PRE_RACE'].fillna(0)
    )

    race_df = race_df.drop(columns=['constructor_standings_points_PRE_RACE'], errors='ignore')
    race_df = race_df.merge(
        constructor_race_points[constructor_points_keys + ['constructor_standings_points_PRE_RACE']],
        on=constructor_points_keys,
        how='left'
    )

    # Constructor PRE positions: rank on unique constructor table, then map back.
    constructor_pre_rank_cols = ['year', 'round', 'constructorId', 'constructor_standings_points_PRE_RACE']
    if 'raceId' in constructor_race_points.columns:
        constructor_pre_rank_cols.insert(2, 'raceId')

    constructor_pre_rank_table = constructor_race_points[constructor_pre_rank_cols].copy()
    constructor_pre_rank_table['constructor_standings_position_PRE_RACE'] = (
        constructor_pre_rank_table.groupby(['year', 'round'])['constructor_standings_points_PRE_RACE']
        .rank(method='min', ascending=False)
        .astype("Int64")
    )

    race_df = race_df.drop(columns=['constructor_standings_position_PRE_RACE'], errors='ignore')
    race_df = race_df.merge(
        constructor_pre_rank_table[[c for c in constructor_points_keys if c in constructor_pre_rank_table.columns] + ['constructor_standings_position_PRE_RACE']],
        on=constructor_points_keys,
        how='left'
    )

    # Ensure season opener stays missing for PRE_RACE positions
    first_round_by_constructor = race_df.groupby(['year', 'constructorId'])['round'].transform('min')
    race_df.loc[
        race_df['round'] == first_round_by_constructor,
        'constructor_standings_position_PRE_RACE'
    ] = pd.NA
    
    race_df = race_df.drop(columns=['standings_points_delta'], errors='ignore')

    print(f"  ✓ Calculated driver and constructor standings (race + sprint points)")
    print(f"  ✓ Calculated PRE_RACE versions")
    
    return race_df

## Ensure constructorId exists before standings
race_data_2024['constructorId'] = race_data_2024.apply(get_constructor_id, axis=1)

print("Testing standings calculation for 2024:")
race_data_2024 = calculate_standings(race_data_2024)
print(f"\nSample with standings:")
event_col = 'Event' if 'Event' in race_data_2024.columns else ('name' if 'name' in race_data_2024.columns else None)
preview_cols = ['year'] + ([event_col] if event_col else []) + ['code', 'constructorId', 'points',
                      'driver_standings_points', 'driver_standings_position',
                      'constructor_standings_points', 'constructor_standings_position']
preview_cols = [c for c in preview_cols if c in race_data_2024.columns]
print(race_data_2024[preview_cols].head(10))


Testing standings calculation for 2024:
Calculating standings...
  ✓ Calculated driver and constructor standings (race + sprint points)
  ✓ Calculated PRE_RACE versions

Sample with standings:
   year               Event code  constructorId  points  \
0  2024  Bahrain Grand Prix  VER              9    26.0   
1  2024  Bahrain Grand Prix  PER              9    18.0   
2  2024  Bahrain Grand Prix  SAI              6    15.0   
3  2024  Bahrain Grand Prix  LEC              6    12.0   
4  2024  Bahrain Grand Prix  RUS            131    10.0   
5  2024  Bahrain Grand Prix  NOR              1     8.0   
6  2024  Bahrain Grand Prix  HAM            131     6.0   
7  2024  Bahrain Grand Prix  PIA              1     4.0   
8  2024  Bahrain Grand Prix  ALO            117     2.0   
9  2024  Bahrain Grand Prix  STR            117     1.0   

   driver_standings_points  driver_standings_position  \
0                     26.0                          1   
1                     18.0                 

## Additional Derived Features


In [72]:
def add_derived_features(race_df):
    """
    Add derived features: podium, status_category (already calculated), etc.
    Note: Rolling features will be calculated separately after appending to master.
    """
    if len(race_df) == 0:
        race_df['podium'] = pd.NA
        return race_df
    
    print("Adding derived features...")
    
    # Podium: 1 if position in [1,2,3], else 0
    race_df['podium'] = race_df['position'].apply(lambda x: 1 if pd.notna(x) and x in [1, 2, 3] else 0)
    
    # status_category already calculated in extract_race_data
    
    print(f"  ✓ Added podium indicator")
    print(f"  ✓ status_category already calculated")
    
    return race_df

# Test derived features for 2024
print("Testing derived features for 2024:")
race_data_2024 = add_derived_features(race_data_2024)
print(f"\nSample with derived features:")
print(race_data_2024[['year', 'Event', 'code', 'position', 'podium', 'status_category']].head(10))


Testing derived features for 2024:
Adding derived features...
  ✓ Added podium indicator
  ✓ status_category already calculated

Sample with derived features:
   year               Event code  position  podium status_category
0  2024  Bahrain Grand Prix  VER       1.0       1        Finished
1  2024  Bahrain Grand Prix  PER       2.0       1        Finished
2  2024  Bahrain Grand Prix  SAI       3.0       1        Finished
3  2024  Bahrain Grand Prix  LEC       4.0       0        Finished
4  2024  Bahrain Grand Prix  RUS       5.0       0        Finished
5  2024  Bahrain Grand Prix  NOR       6.0       0        Finished
6  2024  Bahrain Grand Prix  HAM       7.0       0        Finished
7  2024  Bahrain Grand Prix  PIA       8.0       0        Finished
8  2024  Bahrain Grand Prix  ALO       9.0       0        Finished
9  2024  Bahrain Grand Prix  STR      10.0       0        Finished


## Column Ordering and Final Structure

Reorder columns to match master_races_clean.csv structure. Select only columns that are being reproduced.


In [73]:
# Get column order from master_races_clean.csv
master_columns = pd.read_csv(PROCESSED_DATA_DIR / 'master_races_clean.csv', nrows=0).columns.tolist()

# Columns we're reproducing (core columns, excluding rolling features which need historical data)
reproduced_columns = [
    'resultId', 'raceId', 'driverId', 'constructorId', 'grid', 'position', 'points', 'laps', 
    'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId',
    'year', 'round', 'circuitId', 'date', 'name', 'lat', 'lng', 'code',
    'driver_standings_points', 'driver_standings_position', 'constructor_standings_points', 
    'constructor_standings_position', 'q1', 'q2', 'q3',
    'sprint_results_grid', 'sprint_results_positionOrder', 'sprint_results_points',
    'sprint_results_laps', 'sprint_results_time', 'sprint_results_milliseconds',
    'sprint_results_fastestLap', 'sprint_results_fastestLapTime', 'sprint_results_statusId',
    'podium', 'driver_standings_points_PRE_RACE', 'driver_standings_position_PRE_RACE',
    'constructor_standings_points_PRE_RACE', 'constructor_standings_position_PRE_RACE', 'driver_age', 'status_category'
]

def reorder_columns(race_df, master_columns, reproduced_columns):
    """Reorder columns to match master structure and select reproduced columns."""
    # Add missing columns as NaN
    for col in reproduced_columns:
        if col not in race_df.columns:
            race_df[col] = pd.NA
    
    # Add resultId and robust raceId mapping
    if 'resultId' not in race_df.columns:
        race_df['resultId'] = pd.NA

    # Ensure raceId column exists before filling
    if 'raceId' not in race_df.columns:
        race_df['raceId'] = pd.NA

    # Synthetic raceId (FastF1-only reproduction): stable per (year, round)
    #
    # We intentionally do not join to Kaggle `races.csv` here. Any parity checks vs
    # `master_races_clean.csv` should merge on natural keys (year/round/driver/etc),
    # not on `raceId`.
    year_num = pd.to_numeric(race_df['year'], errors='coerce')
    round_num = pd.to_numeric(race_df.get('round', pd.NA), errors='coerce')

    synthetic = (year_num * 1000 + round_num).astype('Int64')
    race_df['raceId'] = synthetic

    race_df['raceId'] = pd.to_numeric(race_df['raceId'], errors='coerce').astype('Int64')

    # Ergast `rank`: recompute when fastestLapTime present (default '0.0' sentinel).
    if 'fastestLapTime' in race_df.columns:
        race_df = compute_ergast_rank(race_df)
    elif 'rank' not in race_df.columns:
        race_df['rank'] = '0.0'

    # Select and reorder columns
    available_cols = [col for col in reproduced_columns if col in race_df.columns]
    race_df = race_df[available_cols]
    
    # Reorder to match master column order (for columns that exist in both)
    final_order = [col for col in master_columns if col in race_df.columns]
    final_order.extend([col for col in race_df.columns if col not in final_order])
    race_df = race_df[final_order]
    
    return race_df

# Test column ordering for 2024
print("Reordering columns for 2024:")
race_data_2024 = reorder_columns(race_data_2024, master_columns, reproduced_columns)
print(f"\nFinal columns ({len(race_data_2024.columns)}):")
print(race_data_2024.columns.tolist())
print(race_data_2024.head())
print(f"\nShape: {race_data_2024.shape}")


Reordering columns for 2024:

Final columns (46):
['resultId', 'raceId', 'driverId', 'constructorId', 'grid', 'position', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId', 'year', 'round', 'circuitId', 'date', 'name', 'lat', 'lng', 'code', 'driver_standings_points', 'driver_standings_position', 'constructor_standings_points', 'constructor_standings_position', 'q1', 'q2', 'q3', 'sprint_results_grid', 'sprint_results_positionOrder', 'sprint_results_points', 'sprint_results_laps', 'sprint_results_time', 'sprint_results_milliseconds', 'sprint_results_fastestLap', 'sprint_results_fastestLapTime', 'sprint_results_statusId', 'podium', 'driver_standings_points_PRE_RACE', 'constructor_standings_points_PRE_RACE', 'driver_standings_position_PRE_RACE', 'constructor_standings_position_PRE_RACE', 'status_category', 'driver_age']
  resultId   raceId  driverId  constructorId  grid  position  points  laps  \
0     <NA>  2024001       830   

## Complete Pipeline Function

Combine all steps into a single function for easy reproduction.


In [74]:
def reproduce_fastf1_year(year, circuits_df, drivers_df):
    """
    Complete pipeline to reproduce Kaggle-equivalent data from FastF1 for a given year.
    """
    print(f"\n{'='*80}")
    print(f"REPRODUCING FASTF1 DATA FOR {year}")
    print(f"{'='*80}\n")
    
    # Step 1: Load FastF1 data
    fastf1_data = load_fastf1_year(year)
    
    # Step 2: Extract race data
    race_df = extract_race_data(fastf1_data, year)
    if len(race_df) == 0:
        print(f"⚠ No race data found for {year}")
        return pd.DataFrame()

    # Step 3: Map circuit info (no Kaggle races.csv)
    race_df = map_circuit_info(race_df, circuits_df)

    # Step 4: Map round/date from FastF1 schedule (ALL_EVENT_SCHEDULE.csv), with FastF1-only laps fallback
    race_df = map_event_schedule_info(race_df, event_schedule_df, laps_df=fastf1_data.get("laps"))
    
    # Step 5: Map constructorId (use TeamName mapping)
    race_df['constructorId'] = race_df.apply(get_constructor_id, axis=1)
    
    # Step 6: Calculate fastest lap features
    race_df = calculate_fastest_lap_features(race_df, fastf1_data['laps'])
    
    # Step 7: Calculate fastest lap speed
    race_df = calculate_fastest_lap_speed(race_df, fastf1_data['laps'])
    
    # Step 8: Add driver age
    race_df = add_driver_age(race_df, drivers_df)
    
    # Step 9: Extract sprint results
    race_df = extract_sprint_data(race_df, fastf1_data, year)
    
    # Step 10: Calculate standings
    race_df = calculate_standings(race_df)
    
    # Step 11: Add derived features
    race_df = add_derived_features(race_df)
    
    # Step 12: Reorder columns
    race_df = reorder_columns(race_df, master_columns, reproduced_columns)
    
    print(f"\n✓ Completed reproduction for {year}: {len(race_df)} rows")
    return race_df

print("✓ Complete pipeline function defined")


✓ Complete pipeline function defined


## 2024 Validation

Compare reproduced 2024 data against master_races_clean.csv to verify accuracy.

**Note**: Gap times are automatically converted to absolute race times during data extraction, so milliseconds comparison uses absolute times only.


In [75]:
# Reproduce 2024 data using complete pipeline
reproduced_2024 = reproduce_fastf1_year(2024, circuits_df, drivers_df)

# Load master 2024 data for comparison
master_2024 = pd.read_csv(PROCESSED_DATA_DIR / 'master_races_clean.csv', low_memory=False)
master_2024 = master_2024[master_2024['year'] == 2024].copy()

print(f"\n{'='*80}")
print("2024 VALIDATION")
print(f"{'='*80}\n")

print(f"Row counts:")
print(f"  Master 2024: {len(master_2024)}")
print(f"  Reproduced 2024: {len(reproduced_2024)}")
print(f"  Difference: {len(master_2024) - len(reproduced_2024)}")

# Compare key columns
comparison_cols = ['grid', 'position', 'points', 'laps', 'milliseconds', 'q1', 'q2', 'q3', 
                   'statusId', 'driver_standings_points', 'constructor_standings_points']

print(f"\nColumn comparison (matching on year, name, code):")
# Master uses 'name', reproduced also has 'name' (from map_circuit_info)
master_2024_merge = master_2024[['year', 'name', 'code'] + comparison_cols].copy()
master_2024_merge['name'] = master_2024_merge['name'].astype(str).str.strip()
master_2024_merge['code'] = master_2024_merge['code'].astype(str).str.strip().str.upper()

reproduced_2024_merge = reproduced_2024[['year', 'name', 'code'] + [c for c in comparison_cols if c in reproduced_2024.columns]].copy()
reproduced_2024_merge['name'] = reproduced_2024_merge['name'].astype(str).str.strip()
reproduced_2024_merge['code'] = reproduced_2024_merge['code'].astype(str).str.strip().str.upper()

merged = master_2024_merge.merge(
    reproduced_2024_merge,
    on=['year', 'name', 'code'],
    how='outer',
    suffixes=('_master', '_reproduced'),
    indicator=True
)

print(f"\nMerge results:")
print(f"  Both: {len(merged[merged['_merge'] == 'both'])}")
print(f"  Only master: {len(merged[merged['_merge'] == 'left_only'])}")
print(f"  Only reproduced: {len(merged[merged['_merge'] == 'right_only'])}")

# Compare values for matched rows
matched = merged[merged['_merge'] == 'both'].copy()
print(f"\nDetailed comparison for {len(matched)} matched rows:\n")

for col in comparison_cols:
    if f'{col}_master' in matched.columns and f'{col}_reproduced' in matched.columns:
        master_vals = matched[f'{col}_master'].copy()
        repro_vals = matched[f'{col}_reproduced'].copy()
        
        # Normalize \N to NA for master values
        if master_vals.dtype == 'object':
            master_vals = master_vals.replace('\\N', pd.NA)
            master_vals = master_vals.replace(r'\N', pd.NA)  # Handle escaped version
        
        # Handle position: convert both to int for comparison
        if col == 'position':
            # Convert to numeric, then to int (handles floats)
            master_vals = pd.to_numeric(master_vals, errors='coerce').astype('Int64')  # Nullable int
            repro_vals = pd.to_numeric(repro_vals, errors='coerce').astype('Int64')
        
        # Handle timing columns (q1, q2, q3): normalize format
        if col in ['q1', 'q2', 'q3']:
            def normalize_time_format(time_val):
                """Normalize time to mm:ss.mmm format."""
                if pd.isna(time_val):
                    return pd.NA
                time_str = str(time_val).strip()
                
                # Handle \N
                if time_str in ['\\N', r'\N', 'nan', 'NaT', 'None', '']:
                    return pd.NA
                
                # If already in mm:ss.mmm format, return as is
                if re.match(r'^\d+:\d{2}\.\d+$', time_str):
                    return time_str
                
                # If in timedelta format "0 days 00:01:23.821000", extract mm:ss.mmm
                if 'days' in time_str:
                    # Extract the time part after "days"
                    time_part = time_str.split('days', 1)[1].strip()
                    # Parse to get mm:ss.mmm
                    try:
                        td = pd.to_timedelta(time_part)
                        total_seconds = td.total_seconds()
                        minutes = int(total_seconds // 60)
                        seconds = total_seconds % 60
                        return f"{minutes}:{seconds:05.3f}"
                    except:
                        return pd.NA
                
                return time_str
            
            master_vals = master_vals.apply(normalize_time_format)
            repro_vals = repro_vals.apply(normalize_time_format)
        
        # Handle milliseconds: convert to numeric for comparison
        # Note: Gap times should already be converted to absolute times in extract_race_data
        if col == 'milliseconds':
            # Convert master to numeric (handles \N, strings, etc.)
            master_vals = pd.to_numeric(master_vals, errors='coerce')
            # Repro_vals should already be numeric/int, but ensure it is
            repro_vals = pd.to_numeric(repro_vals, errors='coerce')
        
        # Now do the comparison with normalized values
        both_na = master_vals.isna() & repro_vals.isna()
        master_na_only = master_vals.isna() & repro_vals.notna()
        repro_na_only = master_vals.notna() & repro_vals.isna()
        both_not_na = master_vals.notna() & repro_vals.notna()
        
        # Initialize matches array
        matches = pd.Series([False] * len(matched), index=matched.index)
        
        # Both NA = match
        matches[both_na] = True
        
        # For non-NA values, compare
        if both_not_na.sum() > 0:
            master_not_na = master_vals[both_not_na]
            repro_not_na = repro_vals[both_not_na]
            
            if col in ['q1', 'q2', 'q3']:
                # String comparison for normalized time strings
                string_matches = (master_not_na == repro_not_na)
                matches[both_not_na] = string_matches
            elif master_vals.dtype in ['float64', 'int64', 'Int64'] and repro_vals.dtype in ['float64', 'int64', 'Int64']:
                # Numeric comparison
                if master_vals.dtype == 'float64' or repro_vals.dtype == 'float64':
                    # For milliseconds, use tolerance (1 second = 1000 ms) since times may have small rounding differences
                    if col == 'milliseconds':
                        numeric_matches = abs(master_not_na - repro_not_na) < 1000
                    else:
                        numeric_matches = abs(master_not_na - repro_not_na) < 0.001
                else:
                    numeric_matches = (master_not_na == repro_not_na)
                matches[both_not_na] = numeric_matches
            else:
                # String/object comparison
                string_matches = (master_not_na == repro_not_na)
                string_matches = string_matches.fillna(False)
                matches[both_not_na] = string_matches
        
        # Calculate match rate
        match_rate = matches.sum() / len(matched) * 100
        mismatch_count = (~matches).sum()
        
        print(f"{col}:")
        print(f"  Match rate: {match_rate:.1f}% ({matches.sum()}/{len(matched)})")
        print(f"  Both NA (match): {both_na.sum()}")
        print(f"  Master NA only: {master_na_only.sum()}")
        print(f"  Reproduced NA only: {repro_na_only.sum()}")
        print(f"  Both not NA: {both_not_na.sum()}")
        
        # Show sample mismatches
        if mismatch_count > 0 and mismatch_count <= 20:
            print(f"\n  Sample mismatches ({mismatch_count} total):")
            mismatches = matched[~matches].copy()
            for idx, row in mismatches.head(10).iterrows():
                master_val = row[f'{col}_master']
                repro_val = row[f'{col}_reproduced']
                name = row.get('name', 'Unknown')
                code = row.get('code', 'Unknown')
                print(f"    {name} | {code}: Master={master_val} → Reproduced={repro_val}")
        elif mismatch_count > 20:
            print(f"\n  Sample mismatches (showing first 10 of {mismatch_count}):")
            mismatches = matched[~matches].copy()
            for idx, row in mismatches.head(10).iterrows():
                master_val = row[f'{col}_master']
                repro_val = row[f'{col}_reproduced']
                name = row.get('name', 'Unknown')
                code = row.get('code', 'Unknown')
                print(f"    {name} | {code}: Master={master_val} → Reproduced={repro_val}")
        
        print()  # Blank line between columns

print(f"\n✓ Validation complete")


REPRODUCING FASTF1 DATA FOR 2024

Loading FastF1 data for 2024...
  ✓ RESULTS: 2,277 rows, 6 sessions
  ✓ LAPS: 62,690 rows, 6 sessions
  ⚠ TELEMETRY file not found: ALL_TELEMETRY_2024.csv
Extracting race data from 479 race results...
  ✓ Extracted 479 race entries
  ✓ Unique events: 24
  ✓ Unique drivers: 24
Mapping circuit information...
  ✓ All 479 rows mapped to circuits
Calculating fastest lap features from LAPS data...
  ✓ Found fastest lap for 463/479 drivers
Calculating fastest lap speed from LAPS data...
  ✓ Found fastest lap speed for 463/479 drivers
Calculating driver age...
  ✓ Calculated age for 479/479 drivers
Extracting sprint results...
  ✓ Found sprint results for 120/479 drivers
  ✓ Sprint events this year: 6
Calculating standings...
  ✓ Calculated driver and constructor standings (race + sprint points)
  ✓ Calculated PRE_RACE versions
Adding derived features...
  ✓ Added podium indicator
  ✓ status_category already calculated

✓ Completed reproduction for 2024: 479 

In [76]:
# Core + extended diff harness vs `master_races_clean.csv`
# Merge on natural keys (repro uses synthetic `raceId`, so do not join on `raceId`).
# This is an audit tool; it does not modify the reproduction outputs.

import pandas as pd
import numpy as np
import re

# Default: validate quickly to unblock 2025 appending.
# Set to list(range(2018, 2025)) for full 2018-2024 audit.
YEARS_TO_VALIDATE = [2024]

MERGE_KEYS = ["year", "round", "driverId"]

CORE_COLS = [
    "podium",
    "points",
    "driver_standings_points",
    "driver_standings_position",
    "driver_standings_points_PRE_RACE",
    "driver_standings_position_PRE_RACE",
    "constructor_standings_points",
    "constructor_standings_position",
    "constructor_standings_points_PRE_RACE",
    "constructor_standings_position_PRE_RACE",
]

SKIP_EXTENDED_COLS = {
    "resultId",
    "raceId",
    "driverStandingsId",
    "constructorStandingsId",
    "constructor_standings_wins",
}

EXTENDED_DIFF = True
SHOW_EXTENDED_DETAILS = True
MAX_SAMPLE_PAIRS = 8

TIME_LIKE_COLS = {
    "time",
    "fastestLapTime",
    "sprint_results_time",
    "sprint_results_fastestLapTime",
    "q1",
    "q2",
    "q3",
}


def _to_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def _normalize_obj_na(series):
    s = series.copy()
    if s.dtype == "object":
        s = s.replace({"\\N": pd.NA, r"\N": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA, "": pd.NA})
    return s


def _clean_time_text(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    if s in ("", "\\N", "nan", "NaN", "None"):
        return pd.NA
    s = s.replace("\u202f", "").replace("\xa0", "").strip()

    # strip leading '+'
    if s.startswith("+"):
        s = s[1:].strip()

    # "1 days 00:01:23.456" -> "00:01:23.456"
    if "days" in s:
        try:
            s = s.split("days", 1)[1].strip()
        except Exception:
            pass

    # "MM:SS.mmm" -> "00:MM:SS.mmm"
    if re.match(r"^\d+:\d{2}\.\d{1,6}$", s):
        s = "00:" + s

    return s


def _to_timedelta(series):
    s = _normalize_obj_na(series).map(_clean_time_text)
    return pd.to_timedelta(s, errors="coerce")


def _compare_exact(master_s, repro_s, atol=1e-9, rtol=0):
    master_s = _normalize_obj_na(master_s)
    repro_s = _normalize_obj_na(repro_s)
    m = _to_numeric(master_s)
    r = _to_numeric(repro_s)

    both_na = m.isna() & r.isna()
    match = both_na.copy()

    both_not_na = (~m.isna()) & (~r.isna())
    if both_not_na.sum() > 0:
        match.loc[both_not_na] = np.isclose(
            m.loc[both_not_na], r.loc[both_not_na], atol=atol, rtol=rtol
        )
    return match


def _compare_relaxed(master_s, repro_s, col: str) -> pd.Series:
    """Row-wise match for extended columns (mixed dtypes / time strings / dates)."""
    m = _normalize_obj_na(master_s)
    r = _normalize_obj_na(repro_s)

    both_na = m.isna() & r.isna()
    one_na = m.isna() ^ r.isna()
    b = m.notna() & r.notna()
    match = both_na.copy()

    if b.sum() == 0:
        return match & ~one_na

    if col == "date" or col.endswith("_date"):
        md = pd.to_datetime(m, errors="coerce").dt.normalize()
        rd = pd.to_datetime(r, errors="coerce").dt.normalize()
        match.loc[b] = md.loc[b].astype("datetime64[ns]") == rd.loc[b].astype("datetime64[ns]")
        return match & ~one_na

    # For time-like columns, compare as timedeltas first
    if col in TIME_LIKE_COLS:
        mt = _to_timedelta(m)
        rt = _to_timedelta(r)
        both_td = b & mt.notna() & rt.notna()
        if both_td.sum() > 0:
            match.loc[both_td] = np.isclose(
                mt.loc[both_td].dt.total_seconds(),
                rt.loc[both_td].dt.total_seconds(),
                atol=0.001,  # 1ms tolerance
                rtol=0
            )
        rem_td = b & ~both_td
        if rem_td.sum() > 0:
            match.loc[rem_td] = (
                m.loc[rem_td].astype(str).str.strip().str.lower()
                == r.loc[rem_td].astype(str).str.strip().str.lower()
            )
        return match & ~one_na

    mn = pd.to_numeric(m, errors="coerce")
    rn = pd.to_numeric(r, errors="coerce")
    both_num = b & mn.notna() & rn.notna()

    atol = 1e-6
    if col in ("milliseconds", "sprint_results_milliseconds"):
        atol = 2.0
    elif col in ("points", "sprint_results_points", "driver_standings_points", "constructor_standings_points"):
        atol = 1e-4
    elif col in ("lat", "lng"):
        atol = 1e-6

    if both_num.sum() > 0:
        match.loc[both_num] = np.isclose(
            mn.loc[both_num], rn.loc[both_num], atol=atol, rtol=0
        )

    rem = b & ~both_num
    if rem.sum() > 0:
        match.loc[rem] = (
            m.loc[rem].astype(str).str.strip().str.lower()
            == r.loc[rem].astype(str).str.strip().str.lower()
        )

    return match & ~one_na


def _extended_detail(master_s, repro_s, col, mismatch_mask):
    """Detailed diagnostics for one extended column."""
    out = {}
    m = _normalize_obj_na(master_s)
    r = _normalize_obj_na(repro_s)

    both_na = m.isna() & r.isna()
    m_only_na = m.isna() & r.notna()
    r_only_na = r.isna() & m.notna()

    out["both_na"] = int(both_na.sum())
    out["master_only_na"] = int(m_only_na.sum())
    out["repro_only_na"] = int(r_only_na.sum())

    # numeric stats
    mn = pd.to_numeric(m, errors="coerce")
    rn = pd.to_numeric(r, errors="coerce")
    num_mask = mismatch_mask & mn.notna() & rn.notna()
    out["num_mismatch_rows"] = int(num_mask.sum())
    if num_mask.any():
        abs_diff = (rn[num_mask] - mn[num_mask]).abs()
        out["avg_abs_diff"] = float(abs_diff.mean())
        out["median_abs_diff"] = float(abs_diff.median())
        out["p95_abs_diff"] = float(abs_diff.quantile(0.95))
        out["max_abs_diff"] = float(abs_diff.max())

    # time stats
    if col in TIME_LIKE_COLS:
        mt = _to_timedelta(m)
        rt = _to_timedelta(r)
        out["master_td_parse_rate_pct"] = 100.0 * float(mt.notna().mean()) if len(mt) else np.nan
        out["repro_td_parse_rate_pct"] = 100.0 * float(rt.notna().mean()) if len(rt) else np.nan

        both_td = mt.notna() & rt.notna()
        out["both_td_rows"] = int(both_td.sum())
        if both_td.any():
            td_equal = np.isclose(
                mt[both_td].dt.total_seconds(),
                rt[both_td].dt.total_seconds(),
                atol=0.001,
                rtol=0
            )
            out["td_equal_rate_pct"] = 100.0 * float(td_equal.mean())

    # type profile (top types)
    def _top_types(s):
        types = s.map(lambda x: type(x).__name__).value_counts(dropna=False).head(3)
        return ", ".join([f"{idx}:{int(val)}" for idx, val in types.items()])

    out["master_types_top"] = _top_types(m)
    out["repro_types_top"] = _top_types(r)

    # sample mismatch pairs
    mm_idx = mismatch_mask[mismatch_mask].index
    sample_idx = list(mm_idx[:MAX_SAMPLE_PAIRS])
    if sample_idx:
        samples = []
        for i in sample_idx:
            mv = m.loc[i]
            rv = r.loc[i]
            samples.append(f"({repr(mv)} -> {repr(rv)})")
        out["sample_pairs"] = "; ".join(samples)

    return out


print(f"\n{'='*90}\nDIFF VS MASTER (merge on {MERGE_KEYS})\n{'='*90}")

PROCESSED_MASTER_PATH = PROCESSED_DATA_DIR / "master_races_clean.csv"
master_all = pd.read_csv(PROCESSED_MASTER_PATH, low_memory=False)

for year in YEARS_TO_VALIDATE:
    print(f"\n--- Year {year} ---")

    master_year = master_all[master_all["year"] == year].copy()
    if year == 2024 and "reproduced_2024" in globals():
        reproduced_year = reproduced_2024.copy()
    else:
        reproduced_year = reproduce_fastf1_year(year, circuits_df, drivers_df)

    missing_keys = [k for k in MERGE_KEYS if k not in master_year.columns or k not in reproduced_year.columns]
    if missing_keys:
        raise ValueError(f"Missing merge keys for diff: {missing_keys}")

    m = master_year.copy()
    r = reproduced_year.copy()
    for k in MERGE_KEYS:
        m[k] = pd.to_numeric(m[k], errors="coerce").astype("Int64")
        r[k] = pd.to_numeric(r[k], errors="coerce").astype("Int64")

    dup_m = int(m.duplicated(MERGE_KEYS).sum())
    dup_r = int(r.duplicated(MERGE_KEYS).sum())
    if dup_m or dup_r:
        print(f"WARNING: duplicate merge keys — master={dup_m}, repro={dup_r} (deduping keep='first')")
        m = m.drop_duplicates(subset=MERGE_KEYS, keep="first")
        r = r.drop_duplicates(subset=MERGE_KEYS, keep="first")

    merged_full = m.merge(r, on=MERGE_KEYS, how="inner", suffixes=("_master", "_reproduced"), validate="one_to_one")
    print(f"Merged rows (inner join on {MERGE_KEYS}): {len(merged_full):,}")

    if len(merged_full) == 0:
        print("No merged rows; skipping this year.")
        continue

    # --- CORE ---
    print(f"\n-- CORE columns ({len(CORE_COLS)}) --")
    year_core_mismatch_total = 0

    for col in CORE_COLS:
        if col not in m.columns or col not in r.columns:
            print(f"  {col}: SKIP (missing on one side)")
            continue

        mcol, rcol = f"{col}_master", f"{col}_reproduced"
        if mcol not in merged_full.columns or rcol not in merged_full.columns:
            print(f"  {col}: SKIP (missing after merge)")
            continue

        master_s = merged_full[mcol]
        repro_s = merged_full[rcol]
        match_mask = _compare_exact(master_s, repro_s)
        mismatches = int((~match_mask).sum())
        year_core_mismatch_total += mismatches

        match_rate = 100.0 * float(match_mask.mean()) if len(match_mask) else float("nan")
        metric_suffix = ""

        m_num = pd.to_numeric(master_s, errors="coerce")
        r_num = pd.to_numeric(repro_s, errors="coerce")
        valid_num = m_num.notna() & r_num.notna()
        mismatch_num = (~match_mask) & valid_num

        if mismatch_num.any():
            abs_diff = (r_num[mismatch_num] - m_num[mismatch_num]).abs()
            avg_abs_diff = float(abs_diff.mean())

            denom = m_num[mismatch_num].abs()
            non_zero = denom > 0
            if non_zero.any():
                avg_pct_diff = float(((abs_diff[non_zero] / denom[non_zero]) * 100).mean())
                metric_suffix = f", avg_abs_diff={avg_abs_diff:.4f}, avg_pct_diff={avg_pct_diff:.2f}%"
            else:
                metric_suffix = f", avg_abs_diff={avg_abs_diff:.4f}, avg_pct_diff=n/a (master is 0)"

        print(f"  {col}: mismatches={mismatches:,} (match rate={match_rate:.2f}%{metric_suffix})")

    print(f"Year {year} CORE total mismatches: {year_core_mismatch_total:,}")
    if year_core_mismatch_total > 0:
        print(f"WARNING: Core diff mismatches detected for {year}: {year_core_mismatch_total:,} (no assertion raised)")

    # --- EXTENDED ---
    if not EXTENDED_DIFF:
        continue

    common = sorted(set(m.columns) & set(r.columns))
    extra_cols = [
        c for c in common
        if c not in MERGE_KEYS and c not in CORE_COLS and c not in SKIP_EXTENDED_COLS
    ]

    print(f"\n-- EXTENDED columns ({len(extra_cols)}) — informational --")

    ext_rows = []
    ext_details = {}

    for col in extra_cols:
        mcol, rcol = f"{col}_master", f"{col}_reproduced"
        if mcol not in merged_full.columns or rcol not in merged_full.columns:
            continue

        master_s = merged_full[mcol]
        repro_s = merged_full[rcol]

        match_mask = _compare_relaxed(master_s, repro_s, col)
        mismatch_mask = ~match_mask

        mismatches = int(mismatch_mask.sum())
        match_rate = 100.0 * float(match_mask.mean()) if len(match_mask) else float("nan")

        # quick numeric diagnostics
        mn = pd.to_numeric(master_s, errors="coerce")
        rn = pd.to_numeric(repro_s, errors="coerce")
        num_mm = mismatch_mask & mn.notna() & rn.notna()
        avg_abs_diff = float((rn[num_mm] - mn[num_mm]).abs().mean()) if num_mm.any() else np.nan

        ext_rows.append((col, mismatches, match_rate, avg_abs_diff))

        if SHOW_EXTENDED_DETAILS and mismatches > 0:
            ext_details[col] = _extended_detail(master_s, repro_s, col, mismatch_mask)

    # print compact table
    ext_rows.sort(key=lambda x: (-x[1], x[0]))
    for col, mismatches, match_rate, avg_abs_diff in ext_rows:
        tag = "OK" if mismatches == 0 else "DIFF"
        if np.isfinite(avg_abs_diff):
            print(f"  [{tag}] {col}: mismatches={mismatches:,} (match rate={match_rate:.2f}%, avg_abs_diff={avg_abs_diff:.4f})")
        else:
            print(f"  [{tag}] {col}: mismatches={mismatches:,} (match rate={match_rate:.2f}%)")

    ext_total = sum(x[1] for x in ext_rows)
    print(f"Year {year} EXTENDED total mismatched cells (sum over cols): {ext_total:,}")
    print("(EXTENDED mismatches are informational — formats / master-only enrichment may differ.)")

    # detailed deep-dive for worst sprint/time columns
    if SHOW_EXTENDED_DETAILS:
        priority_cols = [
            "sprint_results_time",
            "sprint_results_milliseconds",
            "sprint_results_fastestLapTime",
            "sprint_results_fastestLap",
            "sprint_results_laps",
            "sprint_results_statusId",
            "sprint_results_positionOrder",
            "fastestLapTime",
            "time",
            "milliseconds",
            "rank",
        ]
        print("\n-- EXTENDED DETAIL (priority cols with mismatches) --")
        for c in priority_cols:
            if c not in ext_details:
                continue
            d = ext_details[c]
            print(f"\n  {c}")
            print(f"    NA profile: both_na={d.get('both_na',0):,}, master_only_na={d.get('master_only_na',0):,}, repro_only_na={d.get('repro_only_na',0):,}")
            if "avg_abs_diff" in d:
                print(
                    f"    Numeric diff: avg={d['avg_abs_diff']:.4f}, median={d['median_abs_diff']:.4f}, "
                    f"p95={d['p95_abs_diff']:.4f}, max={d['max_abs_diff']:.4f} "
                    f"(rows={d.get('num_mismatch_rows',0):,})"
                )
            if "master_td_parse_rate_pct" in d:
                print(
                    f"    Timedelta parse: master={d['master_td_parse_rate_pct']:.2f}%, "
                    f"repro={d['repro_td_parse_rate_pct']:.2f}%, "
                    f"equal_when_both_parsed={d.get('td_equal_rate_pct',np.nan):.2f}% "
                    f"(both_parsed_rows={d.get('both_td_rows',0):,})"
                )
            print(f"    Types: master[{d.get('master_types_top','n/a')}], repro[{d.get('repro_types_top','n/a')}]")
            if "sample_pairs" in d:
                print(f"    Sample mismatches: {d['sample_pairs']}")

print("\nDiff validation complete.")


DIFF VS MASTER (merge on ['year', 'round', 'driverId'])

--- Year 2024 ---
Merged rows (inner join on ['year', 'round', 'driverId']): 476

-- CORE columns (10) --
  podium: mismatches=0 (match rate=100.00%)
  points: mismatches=0 (match rate=100.00%)
  driver_standings_points: mismatches=0 (match rate=100.00%)
  driver_standings_position: mismatches=184 (match rate=61.34%, avg_abs_diff=2.2772, avg_pct_diff=12.85%)
  driver_standings_points_PRE_RACE: mismatches=21 (match rate=95.59%)
  driver_standings_position_PRE_RACE: mismatches=0 (match rate=100.00%)
  constructor_standings_points: mismatches=0 (match rate=100.00%)
  constructor_standings_position: mismatches=32 (match rate=93.28%, avg_abs_diff=1.7500, avg_pct_diff=18.86%)
  constructor_standings_points_PRE_RACE: mismatches=20 (match rate=95.80%)
  constructor_standings_position_PRE_RACE: mismatches=410 (match rate=13.87%, avg_abs_diff=4.8634, avg_pct_diff=43.65%)
Year 2024 CORE total mismatches: 667

-- EXTENDED columns (31) — inf

## 2025 Production

Produce 2025 data using the same pipeline.


In [77]:
# Reproduce 2025 data
reproduced_2025 = reproduce_fastf1_year(2025, circuits_df, drivers_df)

# Save to CSV
output_file = PROCESSED_DATA_DIR / 'fastf1_2025_reproduced.csv'
reproduced_2025.to_csv(output_file, index=False)
print(f"\n✓ Saved 2025 data to: {output_file}")
print(f"  Shape: {reproduced_2025.shape}")
print(f"  Columns: {len(reproduced_2025.columns)}")



REPRODUCING FASTF1 DATA FOR 2025

Loading FastF1 data for 2025...
  ⚠ RESULTS file not found: ALL_RESULTS_2025.csv
  ⚠ LAPS file not found: ALL_LAPS_2025.csv
  ⚠ TELEMETRY file not found: ALL_TELEMETRY_2025.csv
⚠ No race data found for 2025

✓ Saved 2025 data to: C:\Users\Erik Viljamaa\Downloads\projects\f1-podium-predictor\data\processed\fastf1_2025_reproduced.csv
  Shape: (0, 0)
  Columns: 0


## Append 2025 to Master and Save

**Note**: Rolling features will need recalculation after appending.


In [78]:
# Load master data
master_full = pd.read_csv(PROCESSED_DATA_DIR / 'master_races_clean.csv', low_memory=False)

# Ensure reproducible IDs for 2025 append
reproduced_2025 = reproduced_2025.copy()

# 1) raceId: map existing IDs when possible; if missing, generate sequential IDs after current max
master_max_raceid = int(pd.to_numeric(master_full['raceId'], errors='coerce').max())
reproduced_2025['raceId'] = pd.to_numeric(reproduced_2025.get('raceId', pd.NA), errors='coerce').astype('Int64')

missing_race_mask = reproduced_2025['raceId'].isna()
if missing_race_mask.any():
    race_keys = ['year', 'round', 'name']
    if 'date' in reproduced_2025.columns:
        race_keys = ['year', 'round', 'date', 'name']

    unique_missing_races = (
        reproduced_2025.loc[missing_race_mask, race_keys]
        .drop_duplicates()
        .sort_values(['year', 'round'] + (['date'] if 'date' in race_keys else []))
        .reset_index(drop=True)
    )
    unique_missing_races['raceId_generated'] = range(master_max_raceid + 1, master_max_raceid + 1 + len(unique_missing_races))

    reproduced_2025 = reproduced_2025.merge(unique_missing_races, on=race_keys, how='left')
    reproduced_2025['raceId'] = reproduced_2025['raceId'].fillna(reproduced_2025['raceId_generated'])
    reproduced_2025 = reproduced_2025.drop(columns=['raceId_generated'], errors='ignore')

reproduced_2025['raceId'] = pd.to_numeric(reproduced_2025['raceId'], errors='coerce').astype('Int64')

# 2) resultId: always generate new sequential IDs after current max for appended rows
master_max_resultid = int(pd.to_numeric(master_full['resultId'], errors='coerce').max())
reproduced_2025 = reproduced_2025.sort_values(['year', 'round', 'raceId', 'driverId']).reset_index(drop=True)
reproduced_2025['resultId'] = range(master_max_resultid + 1, master_max_resultid + 1 + len(reproduced_2025))
reproduced_2025['resultId'] = pd.to_numeric(reproduced_2025['resultId'], errors='coerce').astype('Int64')

# Append 2025 data
master_with_2025 = pd.concat([master_full, reproduced_2025], ignore_index=True)

# Save
# 03.1 expects `master_races_with_2025.csv`, but we also keep the existing `master_races_clean_with_2025.csv` for compatibility.
output_file = PROCESSED_DATA_DIR / 'master_races_with_2025.csv'
master_with_2025.to_csv(output_file, index=False)

output_file_clean = PROCESSED_DATA_DIR / 'master_races_clean_with_2025.csv'
master_with_2025.to_csv(output_file_clean, index=False)

print(f"✓ Appended 2025 data to master")
print(f"  Original master: {len(master_full)} rows")
print(f"  With 2025: {len(master_with_2025)} rows")
print(f"  New raceId range used: {master_max_raceid + 1}+")
print(f"  New resultId range used: {master_max_resultid + 1}..{master_max_resultid + len(reproduced_2025)}")
print(f"  Saved to: {output_file}")
print(f"\n⚠ NOTE: Rolling + PRE_RACE features must be rebuilt after append.")
print("Run the pipeline again in order:")
print("  - 03.1_data_augmentation_v2.ipynb")
print("  - 03.2_feature_engineering.ipynb")
print("  - 03.3_feature_refinement.ipynb")
print("  - 03.4_missing_value_handling.ipynb")
print("  - 03.6_feature_sanity_checks.ipynb")


TypeError: data type 'Int64' not understood

In [ ]:
# Proof: constructor ranking on duplicated driver rows vs unique constructor rows

tmp = reproduced_2024.copy()
for c in ['year','round','constructorId','constructor_standings_points','driverId']:
    if c in tmp.columns:
        tmp[c] = pd.to_numeric(tmp[c], errors='coerce')

# A) current-style (wrong shape): rank on driver rows
a = tmp[['year','round','driverId','constructorId','constructor_standings_points']].copy()
a['rank_driver_rows_min'] = (
    a.groupby(['year','round'])['constructor_standings_points']
     .rank(method='min', ascending=False)
     .astype('Int64')
)

# B) correct shape: rank on unique constructors, then map back
b = (
    tmp[['year','round','constructorId','constructor_standings_points']]
    .drop_duplicates(['year','round','constructorId'])
    .copy()
)
b['rank_unique_constructor_min'] = (
    b.groupby(['year','round'])['constructor_standings_points']
     .rank(method='min', ascending=False)
     .astype('Int64')
)

cmp = a.merge(
    b[['year','round','constructorId','rank_unique_constructor_min']],
    on=['year','round','constructorId'],
    how='left'
)

cmp['different'] = cmp['rank_driver_rows_min'] != cmp['rank_unique_constructor_min']

print("Rows where rank differs:", int(cmp['different'].sum()), "out of", len(cmp))

# show a few races
for rnd in sorted(cmp['round'].dropna().unique())[:5]:
    s = cmp[cmp['round'] == rnd][[
        'round','driverId','constructorId','constructor_standings_points',
        'rank_driver_rows_min','rank_unique_constructor_min'
    ]].sort_values(['rank_unique_constructor_min','constructorId','driverId'])
    print(f"\n--- Round {int(rnd)} sample ---")
    display(s.head(20))

Rows where rank differs: 431 out of 479

--- Round 1 sample ---


,round,driverId,constructorId,constructor_standings_points,rank_driver_rows_min,rank_unique_constructor_min
1,1,815,9,44.0,1,1
0,1,830,9,44.0,1,1
2,1,832,6,27.0,3,2
3,1,844,6,27.0,3,2
6,1,1,131,16.0,5,3
4,1,847,131,16.0,5,3
5,1,846,1,12.0,7,4
7,1,857,1,12.0,7,4
8,1,4,117,3.0,9,5
9,1,840,117,3.0,9,5



--- Round 2 sample ---


,round,driverId,constructorId,constructor_standings_points,rank_driver_rows_min,rank_unique_constructor_min
21,2,815,9,87.0,1,1
20,2,830,9,87.0,1,1
22,2,844,6,49.0,3,2
26,2,860,6,49.0,3,2
27,2,846,1,28.0,5,3
23,2,857,1,28.0,5,3
28,2,1,131,26.0,7,4
25,2,847,131,26.0,7,4
24,2,4,117,13.0,9,5
38,2,840,117,13.0,9,5



--- Round 3 sample ---


,round,driverId,constructorId,constructor_standings_points,rank_driver_rows_min,rank_unique_constructor_min
44,3,815,9,97.0,1,1
58,3,830,9,97.0,1,1
40,3,832,6,93.0,3,2
41,3,844,6,93.0,3,2
42,3,846,1,55.0,5,3
43,3,857,1,55.0,5,3
57,3,1,131,26.0,7,4
56,3,847,131,26.0,7,4
47,3,4,117,25.0,9,5
45,3,840,117,25.0,9,5



--- Round 4 sample ---


,round,driverId,constructorId,constructor_standings_points,rank_driver_rows_min,rank_unique_constructor_min
60,4,815,9,141.0,1,1
59,4,830,9,141.0,1,1
61,4,832,6,120.0,3,2
62,4,844,6,120.0,3,2
63,4,846,1,69.0,5,3
66,4,857,1,69.0,5,3
67,4,1,131,34.0,7,4
65,4,847,131,34.0,7,4
64,4,4,117,33.0,9,5
70,4,840,117,33.0,9,5



--- Round 5 sample ---


,round,driverId,constructorId,constructor_standings_points,rank_driver_rows_min,rank_unique_constructor_min
81,5,815,9,195.0,1,1
79,5,830,9,195.0,1,1
83,5,832,6,151.0,3,2
82,5,844,6,151.0,3,2
80,5,846,1,96.0,5,3
86,5,857,1,96.0,5,3
87,5,1,131,52.0,7,4
84,5,847,131,52.0,7,4
85,5,4,117,40.0,9,5
93,5,840,117,40.0,9,5
